# Laboratorio 1: regresión lineal

#### _Integrantes: Pedro Pablo Sanín (202221527)_

## 1. Exploración de datos

### 1.1 Importación de librerías

Inicialmente importamos todas las librerías que necesitaremos para el modelo.

In [2248]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

### 1.2 Carga del dataset

A continuación cargamos el dataset con los datos metereologicos. Tenemos dos conjuntos de datos: Datos Lab 1.csv para el entrenamiento del modelo, y Datos Test Lab 1.csv para las pruebas del modelo.


In [2249]:
data_meteo = pd.read_csv("./data/Datos Lab 1.csv") #cargamos el dataset Datos Lab 1.csv
data = data_meteo.copy() #hacemos una copia del dataset para no modificar el original


A continuación observamos los primeros registros de los datos, el tamaño del conjunto de datos y el número de columnas

In [2250]:
print("Número de filas y columnas:", data.shape)
data.head()


Número de filas y columnas: (2576, 27)


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.7786,...,-0.3618,-0.0223,183.5308,143.0,2009.0,1.0,invierno,January,S,-2.12
1,2009-01-02,999.6006,997.93,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.4195,...,0.4267,0.3689,40.8436,144.0,2009.0,2.0,invierno,JULY,NE,-0.82
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.2509,...,-0.6993,-0.5268,216.9916,144.0,2009.0,3.0,invierno,January,SO,-0.63
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,97.946704,NaN,4.4904,1.7204,...,-1.1268,-1.0413,222.7419,144.0,2009.0,4.0,invierno,JANUARY,SO,-1.44
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.8003,...,2.6275,0.2874,6.2418,NaN,2009.0,5.0,East,January,N,-10.88


Ahora, obtenemos el tipo de dato de cada una de las columnas. Además, obtendremos la descripción teórica de todos las columnas.

In [2251]:
data.info() #mostramos la información del dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2576 entries, 0 to 2575
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              2504 non-null   object 
 1   presion_media      2501 non-null   float64
 2   presion_min        2502 non-null   float64
 3   presion_max        2512 non-null   float64
 4   presion_desv       2496 non-null   float64
 5   humedad_media      2502 non-null   float64
 6   humedad_min        2496 non-null   float64
 7   humedad_max        2490 non-null   float64
 8   humedad_desv       2515 non-null   float64
 9   viento_media       2490 non-null   float64
 10  viento_min         2485 non-null   float64
 11  viento_max         2495 non-null   float64
 12  viento_desv        2504 non-null   float64
 13  rafaga_media       2498 non-null   float64
 14  rafaga_min         2504 non-null   float64
 15  rafaga_max         2503 non-null   float64
 16  rafaga_desv        2497 

In [2252]:
diccionario_datos = pd.read_excel("./data/Diccionario de datos.xlsx") #cargamos el diccionario de datos
pd.set_option('display.max_colwidth', None)
diccionario_datos

,variable,tipo,descripcion
0,fecha,texto,"Fecha de la observación, en formato día.mes.año."
1,presion_media,numérico,Presión atmosférica media del día (mbar).
2,presion_min,numérico,Presión atmosférica mínima del día (mbar).
3,presion_max,numérico,Presión atmosférica máxima del día (mbar).
4,presion_desv,numérico,Desviación típica de la presión durante el día (mbar).
5,humedad_media,numérico,Humedad relativa media del día (%).
6,humedad_min,numérico,Humedad relativa mínima del día (%).
7,humedad_max,numérico,Humedad relativa máxima del día (%).
8,humedad_desv,numérico,Desviación típica de la humedad relativa (%).
9,viento_media,numérico,Velocidad media del viento durante el día (m/s).


En este caso, concluimos que todos los datos son numéricos, excepto *fecha*, *estacion_anio*, *mes* y *sector_viento*. Esto es consistente con la descripción del Diccionario de datos. Podemos ver además, que el número de datos no nulos es muy variable entre las columnas, lo que sugiere que hay registros con valores nulos.

### 1.3 Explorando el tipo estadístico de los datos

A continuación describiremos el tipo estadístico de cada columna y su rango válido.



| Variable | Tipo de dato | Clasificación estadística | Rango de validez | Consideraciones |
| -------- | -------- | -------- | -------- | -------- |
| *fecha*    | fecha     | Cualitativa ordinal     | 01.01.2003 - 22.07.2026     | Debe verificarse la consistencia de la cadena en días, meses y años     |
| *presion_media*    | numérico     | Cuantitativa continua     | ≥ 0     | Ninguna     |
| *presion_min*    | numérico     | Cuantitativa continua     | ≥ 0     | Debe ser ≤ presion_media     |
| *presion_max*    | numérico     | Cuantitativa continua     | ≥ 0     | Debe ser ≥ presion_media     |
| *presion_desv*    | numérico     | Cuantitativa continua     | ≥ 0     | Al ser una desviación típica, no puede ser negativa     |
| *humedad_media*    | numérico     | Cuantitativa continua     | [0, 100]     | Es un porcentaje, no debe exceder 100     |
| *humedad_min*    | numérico     | Cuantitativa continua     | [0, 100]     | Debe ser ≤ humedad_media     |
| *humedad_max*    | numérico     | Cuantitativa continua     | [0, 100]     | Debe ser ≥ humedad_media     |
| *humedad_desv*    | numérico     | Cuantitativa continua     | ≥ 0     | No puede ser negativa     |
| *viento_media*    | numérico     | Cuantitativa continua     | ≥ 0     | La velocidad no puede ser negativa     |
| *viento_min*    | numérico     | Cuantitativa continua     | ≥ 0     | Debe ser ≤ viento_media     |
| *viento_max*    | numérico     | Cuantitativa continua     | ≥ 0     | Debe ser ≥ viento_media     |
| *viento_desv*    | numérico     | Cuantitativa continua     | ≥ 0     | No puede ser negativa     |
| *rafaga_media*    | numérico     | Cuantitativa continua     | ≥ 0     | Usualmente ≥ viento_media     |
| *rafaga_min*    | numérico     | Cuantitativa continua     | ≥ 0     | Debe ser ≤ rafaga_media     |
| *rafaga_max*    | numérico     | Cuantitativa continua     | ≥ 0     | Debe ser ≥ rafaga_media y ≥ viento_max     |
| *rafaga_desv*    | numérico     | Cuantitativa continua     | ≥ 0     | No puede ser negativa     |
| *viento_norte*    | numérico     | Cuantitativa continua     | Sin límite fijo (puede ser negativo)     | Signo indica sentido norte (+) o sur (−)     |
| *viento_este*    | numérico     | Cuantitativa continua     | Sin límite fijo (puede ser negativo)     | Signo indica sentido este (+) u oeste (−)     |
| *direccion_viento*    | numérico      | Cuantitativa continua     | [0, 360)     | Variable circular; 0° y 360° representan el mismo punto     |
| *registros_del_dia*    | numérico (entero)     | Cuantitativa discreta     | [1, 144]     | Un día completo corresponde a 144 mediciones de 10 minutos     |
| *anio*    | numérico (entero)     | Cuantitativa discreta     | 2003 - 2026     | Debe coincidir con el año contenido en la variable fecha     |
| *dia_del_anio*    | numérico (entero)     | Cuantitativa discreta     | [1, 366]     | El valor máximo depende de si el año es bisiesto     |
| *estacion_anio*    | categórico     | Cualitativa ordinal     | {invierno, primavera, verano, otoño}     | Sigue el orden cíclico natural de las estaciones     |
| *mes*    | categórico     | Cualitativa ordinal     | {enero, febrero, ..., diciembre}     | Debe coincidir con el mes contenido en la variable fecha     |
| *sector_viento*    | categórico     | Cualitativa nominal     | {N, NE, E, SE, S, SO, O, NO}     | No hay jerarquía entre categorías; debe ser consistente con direccion_viento     |
| *temp_max_manana*    | numérico     | Cuantitativa continua     | Rango razonable según clima local (p. ej. -20°C a 55°C)     | Variable objetivo; corresponde al día siguiente     |

Posteriormente, usamos la función describe para conocer la estructura estadística de los datos, tanto cuantitativos como cualitativos. Iniciamos por los datos cuantitativos.


In [2253]:
data.describe() #obtenemos los estadísticos descriptivos del dataset para las variables cuantitativas.

,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,viento_min,...,rafaga_min,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,temp_max_manana
count,2501.000000,2502.000000,2512.000000,2496.000000,2502.000000,2496.000000,2490.000000,2515.000000,2490.000000,2485.000000,...,2504.000000,2503.000000,2497.000000,2496.000000,2512.000000,2497.000000,2499.000000,2490.000000,2508.000000,2481.000000
mean,1010.494271,986.684409,991.704244,1.752041,52.413586,45.094067,90.236582,10.606821,3.814103,1.417799,...,-2.639322,6.946058,-5.763253,-1.128463,-0.410159,172.104573,144.090836,2011.995582,182.690989,13.863309
std,435.334494,8.927138,7.726906,1.215931,35.975384,30.196163,10.647237,5.481338,3.307402,3.960650,...,199.861693,3.237379,75.258134,25.065084,1.147218,86.102348,4.107365,1.999694,105.529469,9.296199
min,948.996000,913.600000,951.940000,0.226900,0.004663,0.237700,27.020000,0.000000,0.000000,0.000000,...,-9999.000000,0.000000,-232.260000,-1250.463300,-3.718500,0.000000,90.000000,2009.000000,1.000000,-15.230000
25%,984.287800,981.371333,987.070000,0.899050,0.869060,1.011790,88.025000,6.082100,1.655350,0.230000,...,0.440000,4.803198,1.002100,-1.640625,-1.236475,91.303500,144.000000,2010.000000,91.000000,7.310000
50%,989.452800,987.070000,991.935000,1.444550,68.558100,49.670000,93.300000,10.199800,2.463400,0.440000,...,0.660000,6.690000,1.461100,-0.564300,-0.356000,199.568500,144.000000,2012.000000,182.500000,14.310000
75%,994.469000,992.567500,996.782500,2.254800,80.562600,67.665000,96.600000,14.722200,4.880505,0.960000,...,1.120000,8.850000,2.008800,0.241525,0.316275,227.101700,144.000000,2014.000000,274.250000,20.460000
max,9939.918000,1012.320198,1013.940000,12.397500,100.000000,103.311405,100.000000,26.341600,24.753600,45.184956,...,19.895916,23.500000,3471.027800,4.809900,4.713900,359.796700,287.000000,2015.000000,366.000000,63.050000


En este caso podemos ver ciertos datos problemáticos. 

1. Vemos primero que *presion_media* tiene una media mayor a la media de *presion_max*, y que además, tiene una desviación estándar demasiado alta y un valor máximo extremadamente alto. Esto nos sugiere que hay un error en los datos, pues el valor máximo de la variable -- 9939.918 mbar -- no es un valor que se de naturalmente ne la tierra. Además, de los cuartiles, puede verse que lo más probable es que este valor simplemente tanga un 9 extra al inicio.
2. Tanto *rafaga_media* como *rafaga_min* alcanzan valores negativos. Esto no se puede pues estos valores no nos dicen nada sobre la maginitud de los vientos. Más aún, los cuartiles están concentrados de tal forma que los datos empiezan en cero, por lo que estos outliers son seguramente errores de medición.
3. *rafaga_desv* tiene un valor mínimo negativo, pese a que la desviación estándar es siempre un número no negativo. Esto también hace que la media de la variable sea también negativa.

A continuación usamos la funcion describe(include='object') para conocer los estadísticos de los datos cualitativos. 


In [2254]:
data.describe(include='object') #obtenemos los estadísticos descriptivos del dataset para todas las variables.

,fecha,estacion_anio,mes,sector_viento
count,2504,2488,2488,2505
unique,2486,28,61,33
top,2010-08-14,primavera,May,SO
freq,2,517,127,695


Ahora podemos ver varios problemas:
1. Hay una fecha repetida, lo que nos indica que puede haber registros repetidos.
2. Hay 28 valores diferentes de *estacion_anio*, pero debería haber 4. Es posible que haya ortografías diferentes.
3. Hay 33 valores diferentes de *sector_viento* pero debrería haber 8 (N,S,E,O,NE,NO,SE,SO). 
4. Hay 61 valores diferentes de *mes* pero debería haber 12.

### 1.4 Valores nulos

Primero obtendremos los valores nulos o vacíos de cada uno de los datos

In [2255]:
columns = data.columns.tolist() #obtenemos la lista de columnas del dataset

for column in columns:
    nulos = data[data[column].isna()].copy() #mostramos los valores nulos de cada columna del dataset
    print(f"Columna: {column}, Números de datos nulos: {len(nulos)}, Proporción de datos nulos: {len(nulos)/len(data)*100:.2f}%")

Columna: fecha, Números de datos nulos: 72, Proporción de datos nulos: 2.80%
Columna: presion_media, Números de datos nulos: 75, Proporción de datos nulos: 2.91%
Columna: presion_min, Números de datos nulos: 74, Proporción de datos nulos: 2.87%
Columna: presion_max, Números de datos nulos: 64, Proporción de datos nulos: 2.48%
Columna: presion_desv, Números de datos nulos: 80, Proporción de datos nulos: 3.11%
Columna: humedad_media, Números de datos nulos: 74, Proporción de datos nulos: 2.87%
Columna: humedad_min, Números de datos nulos: 80, Proporción de datos nulos: 3.11%
Columna: humedad_max, Números de datos nulos: 86, Proporción de datos nulos: 3.34%
Columna: humedad_desv, Números de datos nulos: 61, Proporción de datos nulos: 2.37%
Columna: viento_media, Números de datos nulos: 86, Proporción de datos nulos: 3.34%
Columna: viento_min, Números de datos nulos: 91, Proporción de datos nulos: 3.53%
Columna: viento_max, Números de datos nulos: 81, Proporción de datos nulos: 3.14%
Colum

Ahora, tomaremos la columna *fecha* para determinar como se ven los registros nulos.

In [2256]:
# Filtrar filas con fecha nula
fecha_nulls = data[data['fecha'].isna()].copy()

print(f"Filas con fecha nula: {fecha_nulls.shape[0]}")
indices = data.index[data['fecha'].isna()]
data[data['fecha'].isna()].copy().reset_index(drop=True)

Filas con fecha nula: 72


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,NaN,973.3820,971.76,975.24,1.0043,77.666900,59.8600,89.0,7.3810,3.630300,...,-2.7277,-2.1815,218.6516,144.0,2009.0,86.0,primavera,March,SO,8.97
1,NaN,984.4544,982.66,985.84,0.8370,71.945300,34.6400,98.2,22.1588,5.718600,...,-0.5018,-0.2385,205.4264,144.0,2009.0,97.0,primavera,January,SO,17.42
2,NaN,988.7032,987.59,990.24,0.6328,71.057200,47.7500,95.9,16.3577,1.659500,...,NaN,-1.3504,269.9310,144.0,2009.0,225.0,verano,August,O,22.05
3,NaN,989.2619,986.16,990.44,1.1311,78.713500,62.2500,91.4,7.4702,1.439400,...,-0.5158,0.0946,169.6097,144.0,2009.0,238.0,verano,February,S,27.47
4,NaN,996.1611,995.20,997.71,0.6025,88.620800,76.8000,97.6,7.1956,1.132900,...,-0.0672,-0.3574,259.3584,144.0,2009.0,267.0,otono,September,O,15.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,NaN,991.9774,989.70,994.19,1.4006,0.608340,0.3905,80.8,14.2611,1.847600,...,0.6842,0.5186,37.1636,144.0,2015.0,217.0,Inverano,August,NE,34.41
68,NaN,983.5712,980.65,989.33,2.4154,70.488800,44.4100,84.7,13.7964,1.543600,...,-0.7062,-0.7043,224.9255,144.0,2015.0,244.0,otono,september,SO,20.03
69,NaN,1004.4446,1002.62,1006.43,1.2320,0.006877,0.4476,94.2,17.3187,1.892400,...,1.0049,1.3243,52.8086,144.0,2015.0,274.0,Otoño,October,NE,17.35
70,NaN,1002.1970,999.08,1005.34,2.0512,78.005500,57.3400,93.7,10.8056,10.956240,...,-2.8320,-0.6315,192.5714,144.0,2015.0,339.0,summer,JANUARY,S,9.80


Haremos lo mismo para otra variable, por ejemplo *viento_media*

In [2257]:
# Filtrar filas con fecha nula
viento_media_nulls = data[data['viento_media'].isna()].copy()

print(f"Filas con fecha nula: {viento_media_nulls.shape[0]}")
indices = data.index[data['viento_media'].isna()]
data[data['viento_media'].isna()].copy().reset_index(drop=True)

Filas con fecha nula: 86


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-17,992.1331,988.23,995.15,2.0551,87.702800,82.500000,93.80,2.8481,NaN,...,-2.0065,0.0543,178.4501,144.0,2009.0,NaN,verano,April,S,5.31
1,2009-01-24,961.1252,952.33,970.98,NaN,0.746343,0.945445,55.83,10.7749,NaN,...,-2.1991,-1.7205,218.0386,144.0,2009.0,24.0,invierno,Enero,SO,6.31
2,2009-03-09,980.0093,975.24,984.44,3.0705,74.428500,56.690000,97.00,11.2405,NaN,...,-1.3009,-2.1577,238.9135,144.0,2009.0,68.0,primavera,junio,SO,5.58
3,2009-03-18,1002.7473,998.71,1005.49,2.3902,63.986000,39.860000,NaN,14.3719,NaN,...,-0.4503,-2.3854,259.3108,144.0,2009.0,77.0,Estacion_Desconocida,March,o,4.91
4,2009-04-03,990.4504,988.25,991.95,1.2454,61.731100,31.720000,93.70,20.8908,NaN,...,-1.2460,0.0341,178.4311,144.0,2009.0,93.0,primavera,Jan,S,19.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,2015-09-27,1004.6985,1001.77,1007.60,1.7027,0.745069,0.593100,93.10,9.7399,NaN,...,0.9815,0.7721,38.1900,144.0,2015.0,270.0,otono,JUNE,NE,15.15
82,2015-11-25,980.1694,978.57,984.37,1.3342,79.626400,70.200000,92.70,7.0062,NaN,...,-2.3779,-0.8319,199.2830,144.0,2015.0,329.0,otono,November,NaN,6.45
83,2015-12-07,1004.9268,1001.52,1007.09,1.6678,0.800981,0.683400,NaN,6.1985,NaN,...,-1.0432,-0.3209,197.1000,144.0,2015.0,341.0,verano,SEPTEMBER,NaN,10.53
84,2015-12-07,1004.9268,1001.52,1007.09,1.6678,0.800981,0.683400,94.80,6.1985,NaN,...,-1.0432,-0.3209,197.1000,144.0,2015.0,341.0,verano,SEPTEMBER,S,10.53


Podemos ver que los registros con *fecha* o *viento_media* nula tienen varias columnas que no son nulas. Por lo que podemos suponer que no hay una forma de caracterizar fácilmente los datos con algún registro nulo.

Vamos a revisar cuantos registros tienen **alguna** columna nula.

In [2258]:
nulls_any = data[data.isnull().any(axis=1)] #mostramos las filas con valores nulos en cualquier columna

print(f"Filas con valores nulos en cualquier columna: {nulls_any.shape[0]}")
print(f"Proporción de filas con valores nulos en cualquier columna: {nulls_any.shape[0]/data.shape[0]*100:.2f}%")

Filas con valores nulos en cualquier columna: 1454
Proporción de filas con valores nulos en cualquier columna: 56.44%


Puede verse que la mayoría de datos tienen algún registro nulo, por lo que debemos imputar los registros nulos con alguna técnica no destructiva.

### 1.5 Valores repetidos

Ahora veremos cuantos datos están repetidos. Veremos el resultado en términos de filas idénticas y de duplicados lógicos (ie. dos filas con la misma fecha)

In [2259]:
n_duplicados = int(data.duplicated(keep=False).sum()) #número de filas idénticas en el dataset
print(f"Número de registros duplicados: {n_duplicados}")



Número de registros duplicados: 8


In [2260]:
# Detalle de los registros
dup_table = (
    data[data['fecha'].duplicated(keep=False)]
    .copy()
    .assign(repeticiones=data.groupby('fecha')['fecha'].transform('size'))
    .sort_values(['repeticiones', 'fecha'], ascending=[False, True])
)

# Mostrar tabla
dup_table

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana,repeticiones
21,2009-01-22,9784.1150,971.48,983.54,3.9853,86.731100,69.1100,97.6,9.7926,10.096920,...,-0.1899,183.9897,144.0,2009.0,22.0,verano,octubre,S,5.770000,2.0
2563,2009-01-22,9784.1150,971.48,983.54,3.9853,86.731100,69.1100,97.6,9.7926,10.096920,...,-0.1899,183.9897,144.0,2009.0,22.0,verano,octubre,S,5.770000,2.0
63,2009-03-05,959.8607,958.67,962.54,0.8964,94.294400,86.0000,97.8,2.8296,1.290800,...,0.1826,9.3166,144.0,2009.0,64.0,primavera,July,N,5.370000,2.0
2574,2009-03-05,959.8607,958.67,962.54,0.8964,NaN,86.0000,97.8,2.8296,1.290800,...,0.1826,9.3166,144.0,2009.0,64.0,primavera,July,N,5.956009,2.0
121,2009-05-02,998.1189,995.99,999.41,0.9205,77.500000,51.0900,99.6,18.9020,1.844400,...,0.4668,23.7431,144.0,NaN,122.0,primavera,october,NaN,20.890000,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,NaN,991.9774,989.70,994.19,1.4006,0.608340,0.3905,80.8,14.2611,1.847600,...,0.5186,37.1636,144.0,2015.0,217.0,Inverano,August,NE,34.410000,NaN
2434,NaN,983.5712,980.65,989.33,2.4154,70.488800,44.4100,84.7,13.7964,1.543600,...,-0.7043,224.9255,144.0,2015.0,244.0,otono,september,SO,20.030000,NaN
2464,NaN,1004.4446,1002.62,1006.43,1.2320,0.006877,0.4476,94.2,17.3187,1.892400,...,1.3243,52.8086,144.0,2015.0,274.0,Otoño,October,NE,17.350000,NaN
2529,NaN,1002.1970,999.08,1005.34,2.0512,78.005500,57.3400,93.7,10.8056,10.956240,...,-0.6315,192.5714,144.0,2015.0,339.0,summer,JANUARY,S,9.800000,NaN


Podemos ver que hay registros con fechas repetidas. En este caso, algunos registros tienen valores faltantes y otros no, pese a ser repetidos. En este caso, tiene sentido conservar el registro con menos columnas nulas.

## 2 Preparación de los datos

Primero lidiaremos con los problemas de consistencia de los datos. 



### 2.1 Estandarización variables cualitativas.

Para empezar, debemos estandarizar los valores de las variables cualitativas. Para ello, obtenemos los valores diferentes de cada una.

In [2261]:
print(f"Valores únicos en la columna 'mes': {data['mes'].unique()}")
print(f"Valores únicos en la columna 'sector_viento': {data['sector_viento'].unique()}")
print(f"Valores únicos en la columna 'estacion_anio': {data['estacion_anio'].unique()}")

Valores únicos en la columna 'mes': ['January' 'JULY' 'JANUARY' 'May' 'Enero' nan 'JUNE' 'Sep' 'April'
 'SEPTEMBER' 'noviembre' 'octubre' 'Sept' 'november' 'april' 'julio'
 'February' 'APRIL' 'Aug' 'febrero' 'septiembre' 'Mar' 'Nov' 'MARCH'
 'february' 'November' 'may' 'july' 'august' 'september' 'July' 'March'
 'October' 'junio' 'october' 'march' 'marzo' 'September' 'Oct' 'DECEMBER'
 'Jan' 'enero' 'NOVEMBER' 'december' 'Jul' 'OCTOBER' 'Jun' 'diciembre'
 'AUGUST' 'June' 'january' 'june' 'Feb' 'August' 'December' 'agosto' 'Apr'
 'mayo' 'abril' 'FEBRUARY' 'MAY' 'Dec']
Valores únicos en la columna 'sector_viento': ['S' 'NE' 'SO' 'N' 'South' 'Sureste' nan 'Suroeste' 'Noreste' 'Sur' 'O'
 'North' 'so' 'Oeste' 'o' 'E' 'NORTHEAST' 'NO' 'NORTH' 'SE' 'Noroeste'
 'SOUTHWEST' 'West' 's' 'WEST' 'SOUTH' 'ne' 'East' 'NORTHWEST' 'no' 'EAST'
 'SOUTHEAST' 'Este' 'Norte']
Valores únicos en la columna 'estacion_anio': ['invierno' 'East' nan 'Estacion_Desconocida' 'verano' 'Inverano'
 'invierno ' 'INVIERNO

Podemos ver que hay muchos valores diferentes, por lo que deberíamos primero normalizar los registros y luego usar un diccionario para eliminar las inconsistencias restantes.

Empezamos por limpiar *mes*

In [2262]:
print(f"Valores únicos en la columna 'mes' antes de la limpieza: {data['mes'].unique()}")

data["mes"]=data["mes"].str.lower().str.strip()#convertimos los valores de la columna mes a minúsculasaugust":"agosto", "october":"octubre", "november":"noviembre", "december":"diciembre"})#reemplazamos los valores abreviados de la columna mes por los valores completos
data["mes"]=data["mes"].replace({"enero":"january", "febrero":"february", "marzo":"march", "abril":"april", "mayo":"may", "junio":"june", "julio":"july", "agosto":"august", "septiembre":"september", "octubre":"october", "noviembre":"november", "diciembre":"december", "jan":"january", "feb":"february", "mar":"march", "apr":"april", "jun":"june", "jul":"july","aug":"august","sept":"september" ,"sep":"september","oct":"october","nov":"november","dec":"december"})#reemplazamos los valores abreviados de la columna mes por los valores completos
data["mes"]=data["mes"].str.capitalize()
print(f"Valores únicos en la columna 'mes' después de la limpieza: {data['mes'].unique()}")


Valores únicos en la columna 'mes' antes de la limpieza: ['January' 'JULY' 'JANUARY' 'May' 'Enero' nan 'JUNE' 'Sep' 'April'
 'SEPTEMBER' 'noviembre' 'octubre' 'Sept' 'november' 'april' 'julio'
 'February' 'APRIL' 'Aug' 'febrero' 'septiembre' 'Mar' 'Nov' 'MARCH'
 'february' 'November' 'may' 'july' 'august' 'september' 'July' 'March'
 'October' 'junio' 'october' 'march' 'marzo' 'September' 'Oct' 'DECEMBER'
 'Jan' 'enero' 'NOVEMBER' 'december' 'Jul' 'OCTOBER' 'Jun' 'diciembre'
 'AUGUST' 'June' 'january' 'june' 'Feb' 'August' 'December' 'agosto' 'Apr'
 'mayo' 'abril' 'FEBRUARY' 'MAY' 'Dec']
Valores únicos en la columna 'mes' después de la limpieza: ['January' 'July' 'May' nan 'June' 'September' 'April' 'November'
 'October' 'February' 'August' 'March' 'December']


In [2263]:
data["mes"] = data["mes"].astype("category")

Seguimos limpiando *sector_vieto*

In [2264]:
print(f"Valores únicos en la columna 'sector_viento' antes de la limpieza: {data['sector_viento'].unique()}")
data["sector_viento"]=data["sector_viento"].str.lower().str.strip()#convertimos los valores de la columna sector_viento a minúsculas
data["sector_viento"]=data["sector_viento"].replace({"norte":"n", "sur":"s", "este":"e", "oeste":"w","noreste":"ne", "sureste":"se", "suroeste":"sw", "noroeste":"nw", "north":"n", "south":"s", "east":"e", "west":"w", "northeast":"ne", "southeast":"se", "northwest":"nw", "southwest":"sw", "no":"nw", "o":"w", "so":"sw"})#reemplazamos los valores abreviados de la columna sector_viento por los valores completos
data["sector_viento"]=data["sector_viento"].str.upper()
print(f"Valores únicos en la columna 'sector_viento' después de la limpieza: {data['sector_viento'].unique()}")


Valores únicos en la columna 'sector_viento' antes de la limpieza: ['S' 'NE' 'SO' 'N' 'South' 'Sureste' nan 'Suroeste' 'Noreste' 'Sur' 'O'
 'North' 'so' 'Oeste' 'o' 'E' 'NORTHEAST' 'NO' 'NORTH' 'SE' 'Noroeste'
 'SOUTHWEST' 'West' 's' 'WEST' 'SOUTH' 'ne' 'East' 'NORTHWEST' 'no' 'EAST'
 'SOUTHEAST' 'Este' 'Norte']
Valores únicos en la columna 'sector_viento' después de la limpieza: ['S' 'NE' 'SW' 'N' 'SE' nan 'W' 'E' 'NW']


In [2265]:
data["sector_viento"]= data["sector_viento"].astype("category")

Finalmente limpiamos *estacion_anio*

In [2266]:
print(f"Valores únicos en la columna 'estacion_anio' antes de la limpieza: {data['estacion_anio'].unique()}")

data["estacion_anio"]=data["estacion_anio"].str.lower().str.strip()#convertimos los valores de la columna estacion_anio a minúsculas
data["estacion_anio"]=data["estacion_anio"].replace({"primavera":"spring", "verano":"summer", "otoño":"autumn", "otono":"autumn", "invierno":"winter", "primav":"spring", "primaveraa":"spring", "berano":"summer", "fall":"autumn", "invernio":"winter", "verno":"summer"})#reemplazamos los valores abreviados de la columna estacion_anio por los valores completos
print(f"Valores únicos en la columna 'estacion_anio' después de la limpieza preliminar: {data['estacion_anio'].unique()}")


#vea que en este caso, tenemos valores que no corresponden a ninguna estación del año, por lo que los dejaremos como nan
data.loc[~data["estacion_anio"].isin(["spring", "summer", "autumn", "winter"]), "estacion_anio"] = np.nan
data["estacion_anio"]=data["estacion_anio"].replace({"spring":"primavera","summer":"verano","autumn":"otono", "winter":"invierno"})

print(f"Valores únicos en la columna 'estacion_anio' después de la limpieza: {data['estacion_anio'].unique()}")

Valores únicos en la columna 'estacion_anio' antes de la limpieza: ['invierno' 'East' nan 'Estacion_Desconocida' 'verano' 'Inverano'
 'invierno ' 'INVIERNO' 'VERANO_INVIERNO' 'Verano' 'primavera' 'Primavera'
 'primaveraa' 'PRIMAVERA' 'berano' 'VERANO' 'verno' 'otono' 'otoño '
 'fall' 'autumn' 'OTONO' 'winter' 'invernio' 'primav' 'spring' 'Otoño'
 'Invierno' 'summer']
Valores únicos en la columna 'estacion_anio' después de la limpieza preliminar: ['winter' 'east' nan 'estacion_desconocida' 'summer' 'inverano'
 'verano_invierno' 'spring' 'autumn']
Valores únicos en la columna 'estacion_anio' después de la limpieza: ['invierno' nan 'verano' 'primavera' 'otono']


In [2267]:
data["estacion_anio"]= data["estacion_anio"].astype("category")

Finalmente cambiaremos el tipo de dato de *fecha*

In [2268]:
data["fecha"] = pd.to_datetime(data["fecha"], errors="coerce")
data["fecha"]

0      2009-01-01
1      2009-01-02
2      2009-01-03
3      2009-01-04
4      2009-01-05
          ...    
2571   2015-05-11
2572   2015-02-03
2573   2012-11-22
2574   2009-03-05
2575   2009-12-08
Name: fecha, Length: 2576, dtype: datetime64[ns]

### 2.2 Estandarización de variables cuantitativas y validez

Ahora estandarizaremos las variables cuantitativas.

**presion_media**

Para presión media nos debemos corregir el mayor máximo mal digitado, reemplazando 9939.918 por 939.918. En general, si la presión es mayor a 900, eliminamos el primer 9. A continuación obtenemos los datos que tienen este problema.

In [2269]:
data.loc[data['presion_media']>= 9000, 'presion_media'] #primero vemos estos datos errados



21      9784.115
662     9928.772
1462    9939.918
2561    9939.918
2562    9928.772
2563    9784.115
Name: presion_media, dtype: float64

In [2270]:
#luego eliminamos el primer digito de esos datos
data.loc[data['presion_media']>= 9000, 'presion_media'] = data.loc[data['presion_media']>= 9900, 'presion_media'].astype(str).str[1:].astype(float)
data.describe()

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,rafaga_min,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,temp_max_manana
count,2504,2499.000000,2502.000000,2512.000000,2496.000000,2502.000000,2496.000000,2490.000000,2515.000000,2490.000000,...,2504.000000,2503.000000,2497.000000,2496.000000,2512.000000,2497.000000,2499.000000,2490.000000,2508.000000,2481.000000
mean,2012-06-29 14:19:10.159744512,989.066803,986.684409,991.704244,1.752041,52.413586,45.094067,90.236582,10.606821,3.814103,...,-2.639322,6.946058,-5.763253,-1.128463,-0.410159,172.104573,144.090836,2011.995582,182.690989,13.863309
min,2009-01-01 00:00:00,928.772000,913.600000,951.940000,0.226900,0.004663,0.237700,27.020000,0.000000,0.000000,...,-9999.000000,0.000000,-232.260000,-1250.463300,-3.718500,0.000000,90.000000,2009.000000,1.000000,-15.230000
25%,2010-09-22 18:00:00,984.253850,981.371333,987.070000,0.899050,0.869060,1.011790,88.025000,6.082100,1.655350,...,0.440000,4.803198,1.002100,-1.640625,-1.236475,91.303500,144.000000,2010.000000,91.000000,7.310000
50%,2012-07-04 12:00:00,989.435000,987.070000,991.935000,1.444550,68.558100,49.670000,93.300000,10.199800,2.463400,...,0.660000,6.690000,1.461100,-0.564300,-0.356000,199.568500,144.000000,2012.000000,182.500000,14.310000
75%,2014-03-30 06:00:00,994.382850,992.567500,996.782500,2.254800,80.562600,67.665000,96.600000,14.722200,4.880505,...,1.120000,8.850000,2.008800,0.241525,0.316275,227.101700,144.000000,2014.000000,274.250000,20.460000
max,2015-12-31 00:00:00,1011.653800,1012.320198,1013.940000,12.397500,100.000000,103.311405,100.000000,26.341600,24.753600,...,19.895916,23.500000,3471.027800,4.809900,4.713900,359.796700,287.000000,2015.000000,366.000000,63.050000
std,NaN,8.317852,8.927138,7.726906,1.215931,35.975384,30.196163,10.647237,5.481338,3.307402,...,199.861693,3.237379,75.258134,25.065084,1.147218,86.102348,4.107365,1.999694,105.529469,9.296199


Puede verse ahora que la desviación estándar es similar a la de *presion_min* y *presion_max*

**presion_min y presion_max**

Estas dos columnas no presentan valores atípicos pero es posible que *presion_min* sea mayor a *presion_max* o viceversa. En este caso, cambiamos ambos valores de lugar.



In [2271]:

data.loc[data["presion_min"]>data["presion_max"],["presion_min","presion_max"]]

,presion_min,presion_max
43,994.906458,988.44
44,1003.735938,989.00
56,997.729414,985.86
70,998.971077,987.30
73,1003.906077,989.35
...,...,...
2479,994.400901,988.36
2526,1003.811416,997.78
2536,998.714406,990.12
2540,1001.974454,998.12


In [2272]:
data.loc[data["presion_min"]>data["presion_max"], ["presion_min","presion_max"]] = data.loc[data["presion_min"]>data["presion_max"], ["presion_max","presion_min"]].to_numpy() #cambiamos los valores de orden intercambiando el orden de filas y reemplazando los valores originales por el vector numpy obtenido.
data.loc[data["presion_min"]>data["presion_max"],["presion_min","presion_max"]]

,presion_min,presion_max


Además, si uno de los dos valores es nulo y *presion_media* es mayor a *presion_max* o menor a *presion_min*, también cambiamos de orden.

In [2273]:
#primero obtenemos los valores para los cuales esto pasa
data.loc[(data["presion_min"].isna()|data["presion_max"].isna())&((data["presion_media"]>data["presion_max"])|(data["presion_media"]<data["presion_min"])),["presion_media","presion_min","presion_max"]]



,presion_media,presion_min,presion_max
233,996.8068,NaN,995.05
1134,1005.3334,1010.996120,NaN
1170,995.0744,NaN,992.49
1204,967.4192,NaN,965.92
1231,990.2098,998.586103,NaN
1599,981.2558,987.363892,NaN
2054,986.1887,NaN,982.05
2513,984.4242,NaN,981.13


In [2274]:
data.loc[(data["presion_min"].isna()|data["presion_max"].isna())&((data["presion_media"]>data["presion_max"])|(data["presion_media"]<data["presion_min"])),["presion_media","presion_min","presion_max"]]=data.loc[(data["presion_min"].isna()|data["presion_max"].isna())&((data["presion_media"]>data["presion_max"])|(data["presion_media"]<data["presion_min"])),["presion_media","presion_max","presion_min"]].to_numpy()
data.loc[(data["presion_min"].isna()|data["presion_max"].isna())&((data["presion_media"]>data["presion_max"])|(data["presion_media"]<data["presion_min"])),["presion_media","presion_min","presion_max"]]

,presion_media,presion_min,presion_max


Finalmente, nos aseguramos de que *presion_media* esté entre *presion_min* y *presion_max*. De lo contrario dejamos *presion_media*= 0.5(*presion_min*+*presion_max*). En este caso, suponemos que los valores de presión están normalmente distribuidos.

In [2275]:
data.loc[(data["presion_media"]<data["presion_min"])|(data["presion_media"]>data["presion_max"]),["presion_media","presion_min","presion_max"]]

,presion_media,presion_min,presion_max
662,928.772,986.97,998.17
1462,939.918,985.61,1001.18
2561,939.918,985.61,1001.18
2562,928.772,986.97,998.17


In [2276]:
data.loc[(data["presion_media"]<data["presion_min"])|(data["presion_media"]>data["presion_max"]),"presion_media"] = 0.5*(data.loc[(data["presion_media"]<data["presion_min"])|(data["presion_media"]>data["presion_max"]),"presion_min"]+data.loc[(data["presion_media"]<data["presion_min"])|(data["presion_media"]>data["presion_max"]),"presion_max"])

**presion_desv**

Esta variable no parece tener valores atípicos.

**humedad_media**

Esta variable no parece tener valores atípicos.

**humedad_min** y **humedad_max**

Una vez más, debemos verificar que *humedad_min*≤*humedad_max* y cambiar los valores de orden de lo contrario. Asimismo, vemos que *humedad_max* > 103 en el máximo, por lo que debemos recortar los datos.

In [2277]:
data["humedad_min"]=data["humedad_min"].clip(0,100)
data["humedad_max"]=data["humedad_max"].clip(0,100)

In [2278]:
data.loc[data["humedad_min"]>data["humedad_max"],["humedad_min","humedad_max"]]

,humedad_min,humedad_max
15,100.000000,81.50
34,100.000000,71.70
116,78.238603,41.86
133,82.567768,38.20
145,92.236348,40.81
...,...,...
2388,87.534353,29.95
2443,98.172022,61.88
2469,95.277791,72.20
2498,100.000000,55.03


In [2279]:
data.loc[data["humedad_min"]>data["humedad_max"], ["humedad_min","humedad_max"]] = data.loc[data["humedad_min"]>data["humedad_max"], ["humedad_max","humedad_min"]].to_numpy() #cambiamos los valores de orden intercambiando el orden de filas y reemplazando los valores originales por el vector numpy obtenido.
data.loc[data["humedad_min"]>data["humedad_max"],["humedad_min","humedad_max"]]

,humedad_min,humedad_max


Similarmente, cambiamos el orden si la *humedad_media* ≥ *humedad_max* o *humedad_media* ≤ *humedad_min* y *humedad_max* o *humedad_min* son nulos.

In [2280]:
data.loc[(data["humedad_min"].isna()|data["humedad_max"].isna())&((data["humedad_media"]>data["humedad_max"])|(data["humedad_media"]<data["humedad_min"])),["humedad_media","humedad_min","humedad_max"]]


,humedad_media,humedad_min,humedad_max
3,89.417400,97.946704,NaN
292,84.456600,100.000000,NaN
505,75.196000,NaN,51.65
1472,0.818668,64.370000,NaN
1630,0.690692,37.290000,NaN
1751,77.720300,92.794130,NaN
2078,77.328100,95.751185,NaN
2122,0.837294,59.340000,NaN


In [2281]:
data.loc[(data["humedad_min"].isna()|data["humedad_max"].isna())&((data["humedad_media"]>data["humedad_max"])|(data["humedad_media"]<data["humedad_min"])),["humedad_media","humedad_min","humedad_max"]]=data.loc[(data["humedad_min"].isna()|data["humedad_max"].isna())&((data["humedad_media"]>data["humedad_max"])|(data["humedad_media"]<data["humedad_min"])),["humedad_media","humedad_max","humedad_min"]].to_numpy()
data.loc[(data["humedad_min"].isna()|data["humedad_max"].isna())&((data["humedad_media"]>data["humedad_max"])|(data["humedad_media"]<data["humedad_min"])),["humedad_media","humedad_min","humedad_max"]]

,humedad_media,humedad_min,humedad_max


Finalmente, nos aseguramos de que *humedad_media* esté entre *humedad_min* y *humedad_max*. De lo contrario dejamos *humedad_media*= 0.5(*humedad_min*+*humedad_max*). En este caso, suponemos que los valores de humedad están normalmente distribuidos.

In [2282]:
data.loc[((data["humedad_media"]<data["humedad_min"])|(data["humedad_media"]>data["humedad_max"])),["humedad_media","humedad_min", "humedad_max"]]

,humedad_media,humedad_min,humedad_max
1,0.920868,86.600000,96.30
8,0.839762,0.928437,59.98
17,0.827548,67.980000,94.30
22,0.780708,58.670000,91.10
23,0.746343,0.945445,55.83
...,...,...,...
2525,0.824587,68.790000,97.90
2534,0.900146,72.200000,98.50
2535,0.797528,71.300000,86.00
2540,0.891076,76.400000,95.50


In [2283]:
data.loc[(data["humedad_media"]<data["humedad_min"])|(data["humedad_media"]>data["humedad_max"]),"humedad_media"] = 0.5*(data.loc[(data["humedad_media"]<data["humedad_min"])|(data["humedad_media"]>data["humedad_max"]),"humedad_min"]+data.loc[(data["humedad_media"]<data["humedad_min"])|(data["humedad_media"]>data["humedad_max"]),"humedad_max"])

*viento_media*
Esta variable no presenta datos atípicos.

**viento_min** y **viento_max**

Una vez más, debemos verificar que *viento_min*≤*viento_max* y cambiar los valores de orden de lo contrario. Asimismo, vemos que *viento_max* > 103 en el máximo, por lo que debemos recortar los datos.

In [2284]:
data.loc[data["viento_min"]>data["viento_max"],["viento_min","viento_max"]]

,viento_min,viento_max
17,34.953763,0.684000
22,12.536035,1.137109
32,9.119714,0.037282
103,23.568189,0.000000
120,9.008852,0.070000
...,...,...
2466,4.807191,0.111847
2490,20.850462,0.469756
2499,21.364477,0.432000
2510,7.495023,0.577875


In [2285]:
data.loc[data["viento_min"]>data["viento_max"], ["viento_min","viento_max"]] = data.loc[data["viento_min"]>data["viento_max"], ["viento_max","viento_min"]].to_numpy() #cambiamos los valores de orden intercambiando el orden de filas y reemplazando los valores originales por el vector numpy obtenido.
data.loc[data["viento_min"]>data["viento_max"],["viento_min","viento_max"]]

,viento_min,viento_max


Similarmente, cambiamos el orden si la *viento_media* ≥ *viento_max* o *viento_media* ≤ *viento_min* y *viento_max* o *viento_min* son nulos.

In [2286]:
data.loc[(data["viento_min"].isna()|data["viento_max"].isna())&((data["viento_media"]>data["viento_max"])|(data["viento_media"]<data["viento_min"])),["viento_media","viento_min","viento_max"]]

,viento_media,viento_min,viento_max
33,1.73772,16.576029,NaN
49,1.74520,8.716911,NaN
405,1.76530,6.152328,NaN
1220,7.17696,31.885929,NaN
1475,1.07020,NaN,0.200000
2308,6.78024,NaN,5.267975
2495,4.35312,NaN,0.756000
2507,3.65650,NaN,2.749046
2509,13.16556,NaN,5.796000
2546,24.42888,NaN,13.816271


In [2287]:
data.loc[(data["viento_min"].isna()|data["viento_max"].isna())&((data["viento_media"]>data["viento_max"])|(data["viento_media"]<data["viento_min"])),["viento_media","viento_min","viento_max"]]=data.loc[(data["viento_min"].isna()|data["viento_max"].isna())&((data["viento_media"]>data["viento_max"])|(data["viento_media"]<data["viento_min"])),["viento_media","viento_max","viento_min"]].to_numpy()
data.loc[(data["viento_min"].isna()|data["viento_max"].isna())&((data["viento_media"]>data["viento_max"])|(data["viento_media"]<data["viento_min"])),["viento_media","viento_min","viento_max"]]

,viento_media,viento_min,viento_max


Finalmente, nos aseguramos de que *viento_media* esté entre *viento_min* y *viento_max*. De lo contrario dejamos *viento_media*= 0.5(*viento_min*+*viento_max*). En este caso, suponemos que los valores de viento están normalmente distribuidos.

In [2288]:
data.loc[((data["viento_media"]<data["viento_min"])|(data["viento_media"]>data["viento_max"])),["viento_media","viento_min", "viento_max"]]

,viento_media,viento_min,viento_max
293,1.17530,0.200,1.081085
329,5.00080,0.360,3.818548
354,2.99380,0.780,2.509662
364,2.70510,0.870,2.684323
510,2.13530,0.300,1.648655
576,2.00740,0.290,1.779930
685,9.84744,4.464,6.199252
707,2.95200,0.840,2.715391
721,1.86740,0.200,1.644794
744,2.82280,0.400,2.262557


In [2289]:
data.loc[(data["viento_media"]<data["viento_min"])|(data["viento_media"]>data["viento_max"]),"viento_media"] = 0.5*(data.loc[(data["viento_media"]<data["viento_min"])|(data["viento_media"]>data["viento_max"]),"viento_min"]+data.loc[(data["viento_media"]<data["viento_min"])|(data["viento_media"]>data["viento_max"]),"viento_max"])
data.loc[((data["viento_media"]<data["viento_min"])|(data["viento_media"]>data["viento_max"])),["viento_media","viento_min", "viento_max"]]

,viento_media,viento_min,viento_max


**viento_desv**

Vemos que esta variable tiene outliers. Vamos a verlos.


In [2290]:
data.loc[data["viento_desv"]>1000]

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
2384,2015-07-13,989.8826,988.77,990.94,0.5409,82.5317,60.5,94.6,10.118,NaN,...,-1250.4633,3.5736,179.8363,144.0,2015.0,NaN,verano,July,S,22.26


Puede verse en este caso que este valor es atípico pues, no tiene valores de viento, el valor de la *rafaga_minima* es un error de digitación y esto tiene efectos en "rafaga_media" y *rafaga_desv*. En este caso, y como solamente se trata de un dato, eliminaremos el registro pues tiene muchas inconsistencias. 

In [2291]:
data = data.drop(2384)

Ahora veremos la nueva descripción de los datos.

In [2292]:
data.describe()

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,rafaga_min,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,temp_max_manana
count,2503,2498.000000,2503.000000,2509.000000,2495.000000,2501.000000,2489.000000,2495.000000,2514.000000,2490.000000,...,2503.000000,2502.000000,2496.000000,2495.000000,2511.000000,2496.000000,2498.000000,2489.000000,2508.000000,2480.000000
mean,2012-06-29 03:41:29.652417024,989.160372,986.241243,992.149825,1.752526,57.400686,43.408565,91.788413,10.607016,3.784867,...,1.354430,6.945425,-7.156198,-0.627728,-0.411746,172.101476,144.090873,2011.994375,182.690989,13.859923
min,2009-01-01 00:00:00,948.996000,913.600000,951.940000,0.226900,0.004663,0.237700,33.610000,0.000000,0.000000,...,0.000000,0.000000,-232.260000,-6.797700,-3.718500,0.000000,90.000000,2009.000000,1.000000,-15.230000
25%,2010-09-22 12:00:00,984.286300,981.190000,987.570000,0.899200,47.440700,1.005897,89.000000,6.081800,1.644848,...,0.440000,4.803198,1.001775,-1.636350,-1.236650,91.076500,144.000000,2010.000000,91.000000,7.307500
50%,2012-07-04 00:00:00,989.446100,986.690000,992.320000,1.444600,70.757000,48.390000,93.700000,10.201200,2.438050,...,0.660000,6.690000,1.461050,-0.564300,-0.356200,199.605600,144.000000,2012.000000,182.500000,14.305000
75%,2014-03-29 12:00:00,994.383375,992.020000,997.120000,2.254800,81.221700,65.120000,96.800000,14.722450,4.829600,...,1.120000,8.850000,2.008275,0.241650,0.315150,227.126225,144.000000,2014.000000,274.250000,20.452500
max,2015-12-31 00:00:00,1011.653800,1009.080000,1013.940000,12.397500,100.000000,100.000000,100.000000,26.341600,24.753600,...,19.895916,23.500000,6.054800,4.809900,4.713900,359.796700,287.000000,2015.000000,366.000000,63.050000
std,NaN,8.023933,8.735411,7.633759,1.215933,33.287966,28.635777,7.432102,5.482420,3.288191,...,2.401112,3.237872,28.621590,1.556433,1.144687,86.119462,4.108187,1.999188,105.529469,9.296544


**rafaga_media**

Luego de la limpieza del registro, esta columna no parece tener valores atípicos.

**rafaga_min** y **rafaga_max**

Una vez más, debemos revisar el orden de los datos.

In [2293]:
data.loc[data["rafaga_min"]>data["rafaga_max"],["rafaga_min","rafaga_max"]]



,rafaga_min,rafaga_max
33,4.271407,0.236121
35,6.672666,0.932056
57,10.078922,1.500000
78,11.031436,0.500000
95,7.744997,0.250000
...,...,...
2501,11.171242,1.080000
2538,9.180290,0.240000
2548,10.186582,0.680000
2554,10.479462,0.640000


In [2294]:
data.loc[data["rafaga_min"]>data["rafaga_max"], ["rafaga_min","rafaga_max"]] = data.loc[data["rafaga_min"]>data["rafaga_max"], ["rafaga_max","rafaga_min"]].to_numpy() #cambiamos los valores de orden intercambiando el orden de filas y reemplazando los valores originales por el vector numpy obtenido.
data.loc[data["rafaga_min"]>data["rafaga_max"],["rafaga_min","rafaga_max"]]

,rafaga_min,rafaga_max


Similarmente, cambiamos el orden si la *rafaga_media* ≥ *rafaga_max* o *rafaga_media* ≤ *rafaga_min* y *rafaga_max* o *rafaga_min* son nulos.

In [2295]:
data.loc[(data["rafaga_min"].isna()|data["rafaga_max"].isna())&((data["rafaga_media"]>data["rafaga_max"])|(data["rafaga_media"]<data["rafaga_min"])),["rafaga_media","rafaga_min","rafaga_max"]]

,rafaga_media,rafaga_min,rafaga_max
107,3.6415,NaN,0.702149
563,3.1142,NaN,0.400000
833,3.9974,10.780034,NaN
1042,1.8382,NaN,0.335540
1121,2.5244,8.233842,NaN
1753,3.6152,13.012602,NaN
2190,1.7735,NaN,0.298258


In [2296]:
data.loc[(data["rafaga_min"].isna()|data["rafaga_max"].isna())&((data["rafaga_media"]>data["rafaga_max"])|(data["rafaga_media"]<data["rafaga_min"])),["rafaga_media","rafaga_min","rafaga_max"]]=data.loc[(data["rafaga_min"].isna()|data["rafaga_max"].isna())&((data["rafaga_media"]>data["rafaga_max"])|(data["rafaga_media"]<data["rafaga_min"])),["rafaga_media","rafaga_max","rafaga_min"]].to_numpy()
data.loc[(data["rafaga_min"].isna()|data["rafaga_max"].isna())&((data["rafaga_media"]>data["rafaga_max"])|(data["rafaga_media"]<data["rafaga_min"])),["rafaga_media","rafaga_min","rafaga_max"]]

,rafaga_media,rafaga_min,rafaga_max


Finalmente, nos aseguramos de que *rafaga_media* esté entre *rafaga_min* y *rafaga_max*. De lo contrario dejamos *rafaga_media*= 0.5(*rafaga_min*+*rafaga_max*). En este caso, suponemos que los valores de rafaga están normalmente distribuidos.

In [2297]:
data.loc[((data["rafaga_media"]<data["rafaga_min"])|(data["rafaga_media"]>data["rafaga_max"])),["rafaga_media","rafaga_min", "rafaga_max"]]

,rafaga_media,rafaga_min,rafaga_max
402,4.0438,1.92,3.585311
722,5.8252,2.44,5.312722
780,5.8675,2.76,5.722827
913,6.7676,2.52,6.008658
1102,4.6167,2.44,4.579504
1482,3.1665,1.74,2.963940
1513,6.3492,3.63,6.021085
1517,4.5853,1.64,4.529795
1542,8.9474,4.73,7.841702
1577,5.2830,1.48,5.126311


In [2298]:
data.loc[(data["rafaga_media"]<data["rafaga_min"])|(data["rafaga_media"]>data["rafaga_max"]),"rafaga_media"] = 0.5*(data.loc[(data["rafaga_media"]<data["rafaga_min"])|(data["rafaga_media"]>data["rafaga_max"]),"rafaga_min"]+data.loc[(data["rafaga_media"]<data["rafaga_min"])|(data["rafaga_media"]>data["rafaga_max"]),"rafaga_max"])
data.loc[((data["rafaga_media"]<data["rafaga_min"])|(data["rafaga_media"]>data["rafaga_max"])),["rafaga_media","rafaga_min", "rafaga_max"]]

,rafaga_media,rafaga_min,rafaga_max


**rafaga_desv**

Vemos que esta columna tiene valores negativos, por lo que debemos conocerlos con más detalle.

In [2299]:
data.loc[data["rafaga_desv"]<0]

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
16,2009-01-17,992.1331,988.23,995.15,2.0551,87.702800,82.500000,93.80,2.8481,NaN,...,-2.0065,0.0543,178.4501,144.0,2009.0,NaN,verano,April,S,5.31
21,2009-01-22,NaN,971.48,983.54,3.9853,86.731100,69.110000,97.60,9.7926,10.09692,...,-2.7229,-0.1899,183.9897,144.0,2009.0,22.0,verano,October,S,5.77
26,2009-01-27,986.1106,980.97,990.97,3.2006,98.995800,NaN,100.00,1.1007,8.05644,...,1.9919,0.9426,25.3231,144.0,2009.0,27.0,verano,November,NE,-1.63
33,2009-02-03,973.8671,972.46,976.35,1.1087,32.126592,0.993185,63.26,9.5014,1.73772,...,-0.1624,-0.0542,198.4735,144.0,2009.0,34.0,verano,April,S,4.89
42,2009-02-12,986.7699,983.22,989.49,2.1569,75.185000,58.270000,92.10,9.6642,7.55532,...,0.3389,-1.4666,283.0129,144.0,2009.0,43.0,verano,March,W,1.48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2562,2010-10-25,992.5700,986.97,998.17,3.4396,79.714900,NaN,94.30,9.2834,5.98716,...,-0.4272,-1.2591,251.2583,144.0,2010.0,298.0,otono,October,W,7.34
2563,2009-01-22,NaN,971.48,983.54,3.9853,86.731100,69.110000,97.60,9.7926,10.09692,...,-2.7229,-0.1899,183.9897,144.0,2009.0,22.0,verano,October,S,5.77
2564,2015-12-07,1004.9268,1001.52,1007.09,1.6678,0.800981,0.683400,94.80,6.1985,NaN,...,-1.0432,-0.3209,197.1000,144.0,2015.0,341.0,verano,September,S,10.53
2565,2013-02-08,984.6369,983.06,987.49,1.3454,0.886056,0.760000,96.80,4.6160,NaN,...,0.0254,-0.2613,275.5529,144.0,2013.0,39.0,verano,September,W,-0.60


Vea que los valores son razonables pero simplemente tienen signo negativo, por lo que tomaremos su valor absoluto.

In [2300]:
data["rafaga_desv"] = data["rafaga_desv"].abs()
data["rafaga_desv"].describe()

count    2496.000000
mean       10.158025
std        27.698381
min         0.000000
25%         1.204425
50%         1.649150
75%         2.345600
max       232.260000
Name: rafaga_desv, dtype: float64

En este caso, tenemos datos válidos.

In [2301]:
data.loc[(data["presion_media"]<data["presion_min"])|(data["presion_media"]>data["presion_max"]),["presion_media","presion_min","presion_max"]]

,presion_media,presion_min,presion_max


**viento_norte**

Esta variable no parece tener datos inválidos.


**viento_este**

Esta variable no parece tener datos inválidos.

**direccion_viento**

Esta variable no parece tener datos inválidos. Sin embargo, debemos verificar si *sector_viento* es consistente con *direccion_viento*

In [2302]:
def obtener_sector_compass(grados):
    """Devuelve el sector de la brújula correspondiente a un ángulo [0, 360)."""
    if pd.isna(grados):
        return np.nan

    if not 0 <= grados < 360:
        raise ValueError("El ángulo debe estar en el intervalo [0, 360).")

    sectores = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    indice = int((grados + 22.5) // 45) % 8

    return sectores[indice]

In [2303]:
data.loc[(data["sector_viento"] != data["direccion_viento"].apply(obtener_sector_compass).astype("category"))&(~data["sector_viento"].isna())&(~data["direccion_viento"].isna())]

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana


Podemos ver que en todos los casos, la dirección y el sector son consistentes.

**registros_del_dia**

Esta variable no parece tener datos inválidos, pero nos aseguraremos de que todas las entradas sean enteras.

In [2304]:
data["registros_del_dia"] = data["registros_del_dia"].astype('Int64')
data["registros_del_dia"].head()


0     143
1     144
2     144
3     144
4    <NA>
Name: registros_del_dia, dtype: Int64

**anio** y **dia_del_anio**

Debemos revisar que estos valores sean consistentes con la fecha. Primero convertiremos todas las entradas a entero.

In [2305]:
data["anio"] = data["anio"].astype("Int64")
data["dia_del_anio"] = data["dia_del_anio"].astype("Int64")
data.loc[:,["anio", "dia_del_anio"]].head()

,anio,dia_del_anio
0,2009,1
1,2009,2
2,2009,3
3,2009,4
4,2009,5


Ahora crearemos dos funciones, una para retornar el par *anio*, *dia_del_anio* de una fecha y la otra para retornar la fecha de un par *anio*, *dia_del_anio* y revisaremos que sean consistentes.

In [2306]:
def obtener_anio_dia(fecha):
    """Retorna el año y el día del año de una fecha pandas datetime."""
    if pd.isna(fecha):
        return pd.NA, pd.NA

    fecha = pd.Timestamp(fecha)
    return fecha.year, fecha.dayofyear


def obtener_fecha(anio, dia_del_anio):
    """Retorna una fecha pandas datetime a partir del año y día del año."""
    if pd.isna(anio) or pd.isna(dia_del_anio):
        return pd.NaT

    return pd.Timestamp(anio, 1, 1) + pd.Timedelta(days=int(dia_del_anio) - 1)

In [2307]:
anio_calculado = data["fecha"].dt.year.astype("Int64")
dia_del_anio_calculado = data["fecha"].dt.dayofyear.astype("Int64")
dia_anio_calculado = data["fecha"].dt.dayofyear.astype("Int64")


inconsistencias_anio =data.loc[
    (data["anio"].fillna(-1) != anio_calculado.fillna(-1))&(~data["anio"].isna())&(~data["fecha"].isna())
]

inconsistenciass_dia_del_anio=data.loc[
    (data["dia_del_anio"].fillna(-1) != dia_del_anio_calculado.fillna(-1))&(~data["dia_del_anio"].isna())&(~data["fecha"].isna())
]

print("Número de filas año inconsistente:", len(inconsistencias_anio))

print("Número de filas dia del año inconsistente:", len(inconsistenciass_dia_del_anio))


Número de filas año inconsistente: 0
Número de filas dia del año inconsistente: 0


De esta forma sabemos que el año, el día del año y la fecha son consistentes.

**temperatura_max_manana**

Si bien esta resultado no parece tener valores inválidos, el valor máximo es demasiado alto, teniendo en cuenta que la tempreatura en Alemania nunca ha subido de 45 grados. Por ende recortaremos el conjunto de datos por arriba.

In [2308]:
data["temp_max_manana"] = data["temp_max_manana"].clip(upper=45)
data["temp_max_manana"].describe()

count    2480.000000
mean       13.832132
std         9.176776
min       -15.230000
25%         7.307500
50%        14.305000
75%        20.452500
max        45.000000
Name: temp_max_manana, dtype: float64

### 2.3 Unicidad de los datos

A continuación revisaremos que datos siguen repetidos luego de nuestras transformaciones.

In [2309]:
# Detalle de los registros
dup_table = (
    data[data['fecha'].duplicated(keep=False)]
    .copy()
    .assign(repeticiones=data.groupby('fecha')['fecha'].transform('size'))
    .sort_values(['repeticiones', 'fecha'], ascending=[False, True])
)

dup_table

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana,repeticiones
21,2009-01-22,NaN,971.48,983.54,3.9853,86.73110,69.1100,97.6,9.7926,10.096920,...,-0.1899,183.9897,144,2009,22,verano,October,S,5.770000,2.0
2563,2009-01-22,NaN,971.48,983.54,3.9853,86.73110,69.1100,97.6,9.7926,10.096920,...,-0.1899,183.9897,144,2009,22,verano,October,S,5.770000,2.0
63,2009-03-05,959.8607,958.67,962.54,0.8964,94.29440,86.0000,97.8,2.8296,1.290800,...,0.1826,9.3166,144,2009,64,primavera,July,N,5.370000,2.0
2574,2009-03-05,959.8607,958.67,962.54,0.8964,NaN,86.0000,97.8,2.8296,1.290800,...,0.1826,9.3166,144,2009,64,primavera,July,N,5.956009,2.0
121,2009-05-02,998.1189,995.99,999.41,0.9205,77.50000,51.0900,99.6,18.9020,1.844400,...,0.4668,23.7431,144,<NA>,122,primavera,October,NaN,20.890000,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,NaT,991.9774,989.70,994.19,1.4006,0.60834,0.3905,80.8,14.2611,1.847600,...,0.5186,37.1636,144,2015,217,NaN,August,NE,34.410000,NaN
2434,NaT,983.5712,980.65,989.33,2.4154,70.48880,44.4100,84.7,13.7964,1.543600,...,-0.7043,224.9255,144,2015,244,otono,September,SW,20.030000,NaN
2464,NaT,1004.4446,1002.62,1006.43,1.2320,47.32380,0.4476,94.2,17.3187,1.892400,...,1.3243,52.8086,144,2015,274,otono,October,NE,17.350000,NaN
2529,NaT,1002.1970,999.08,1005.34,2.0512,78.00550,57.3400,93.7,10.8056,10.956240,...,-0.6315,192.5714,144,2015,339,verano,January,S,9.800000,NaN


Para solucionar este problema, conservaremos el dato que tiene menos columnas nulos.

In [2310]:
data["_nulos"] = data.isna().sum()

data = (
    data.sort_values("_nulos")
        .drop_duplicates(subset="fecha", keep="first")
        .drop(columns="_nulos")
)

dup_table = data[data["fecha"].duplicated(keep=False)]
print("Numero de fechas repetidas: ", len(dup_table))
print("Numero de duplicados exactos:", int(data.duplicated(keep=False).sum()))
print("Número de filas: ", len(data))
data.describe()

Numero de fechas repetidas:  0
Numero de duplicados exactos: 0
Número de filas:  2486


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,rafaga_min,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,temp_max_manana
count,2485,2411.000000,2416.000000,2423.000000,2408.000000,2413.000000,2407.000000,2412.000000,2427.000000,2405.000000,...,2417.000000,2416.000000,2409.000000,2411.000000,2427.000000,2409.000000,2409.0,2403.0,2420.0,2393.000000
mean,2012-06-29 05:15:48.893360128,989.133121,986.224358,992.112105,1.746237,57.364320,43.410533,91.747965,10.620578,3.769508,...,0.846910,7.465085,9.982054,-0.622154,-0.411010,171.828483,144.09423,2011.987516,182.273967,13.789197
min,2009-01-01 00:00:00,948.996000,913.600000,951.940000,0.226900,0.004663,0.237700,33.610000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-6.797700,-3.718500,0.000000,90.0,2009.0,1.0,-15.230000
25%,2010-09-23 00:00:00,984.286800,981.220000,987.550000,0.896850,46.219650,1.006273,89.000000,6.068600,1.643100,...,0.440000,5.170000,1.203300,-1.641950,-1.234850,88.448500,144.0,2010.0,91.0,7.300000
50%,2012-07-01 00:00:00,989.422000,986.660000,992.310000,1.443100,70.605600,48.390000,93.669614,10.213500,2.441200,...,0.640000,7.000000,1.647000,-0.561500,-0.357600,199.568500,144.0,2012.0,182.0,14.360000
75%,2014-03-30 00:00:00,994.360100,991.942500,997.065000,2.242025,81.205500,65.120000,96.700000,14.754300,4.768920,...,1.000000,9.250000,2.334900,0.256650,0.319150,227.101700,144.0,2014.0,273.0,20.460000
max,2015-12-31 00:00:00,1011.653800,1009.080000,1013.940000,12.397500,100.000000,100.000000,100.000000,26.341600,24.753600,...,6.530000,23.500000,232.260000,4.809900,4.713900,359.796700,287.0,2015.0,366.0,45.000000
std,NaN,8.007059,8.718493,7.610344,1.213791,33.301501,28.643558,7.464159,5.497942,3.276926,...,0.675449,3.097386,27.462977,1.559101,1.147203,86.382942,4.18338,2.003289,105.188583,9.088060


### 2.4 Completitud de los datos.

Ahora debemos imputar los valores nulos. Para cada columna, definiremos un procedimiento de imputación. En este caso, estos no hacen parte del pipeline pues estamos usando distintas estrategias de normalización e imputación.

**fecha**

Si la fecha es nula, reconstruimos la fecha a partir de *anio* y *dia_del_anio*

In [2311]:
data

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.8750,94.800000,1.7650,0.77860,...,-0.3618,-0.0223,183.5308,143,2009,1,invierno,January,S,-2.120
1,2009-01-02,999.6006,997.93,1002.65,1.5039,91.450000,86.6000,96.300000,2.7588,1.41950,...,0.4267,0.3689,40.8436,144,2009,2,invierno,July,NE,-0.820
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.3900,93.900000,15.1796,1.25090,...,-0.6993,-0.5268,216.9916,144,2009,3,invierno,January,SW,-0.630
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,NaN,97.946704,4.4904,1.72040,...,-1.1268,-1.0413,222.7419,144,2009,4,invierno,January,SW,-1.440
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.6000,93.200000,5.3922,3.80030,...,2.6275,0.2874,6.2418,<NA>,2009,5,NaN,January,N,-10.880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2552,2015-12-28,1003.5725,1002.35,1004.96,0.7294,87.109700,72.1000,97.700000,7.5170,3.02724,...,-0.6644,-0.0853,187.3193,144,<NA>,362,verano,March,S,9.160
2553,2015-12-29,1002.1544,1000.83,1005.33,1.2520,95.091000,79.4000,99.500000,5.3604,0.86610,...,-0.5544,-0.0871,188.9267,144,2015,363,invierno,December,S,5.650
2554,2015-12-30,1002.9883,997.51,1005.73,2.5524,84.473700,NaN,99.400000,12.9567,2.12740,...,-2.0540,-0.1130,183.1497,144,2015,364,invierno,December,S,2.920
2555,2015-12-31,996.9224,995.01,999.20,1.4003,0.726064,0.4956,97.400000,17.6803,3.11310,...,-3.0401,0.1673,176.8493,144,<NA>,365,invierno,NaN,S,2.720


In [2312]:
filas_fecha_nula = data["fecha"].isna()

print("Fechas faltantes restantes:", data["fecha"].isna().sum())

filas_fecha = data.loc[filas_fecha_nula, ["anio", "dia_del_anio"]]


fechas_imputadas = pd.Series(
    [
        obtener_fecha(fila["anio"], fila["dia_del_anio"])
        for _, fila in filas_fecha.iterrows()
    ],
    index=filas_fecha.index,
    dtype="datetime64[ns]"
)

data.loc[filas_fecha_nula, "fecha"] = fechas_imputadas

print("Fechas faltantes restantes:", data["fecha"].isna().sum())


Fechas faltantes restantes: 1
Fechas faltantes restantes: 0


**registros_del_dia**

Como la mayoría de datos tienen 144 registros, llenamos los registros nulos con este valor.

In [2313]:
filas_registros_del_dia_null = data["registros_del_dia"].isna()


print(
    "Registros del dia faltantes restantes:",
    data["registros_del_dia"].isna().sum()
)

data.loc[filas_registros_del_dia_null,"registros_del_dia"] = data["registros_del_dia"].mode()[0]

print(
    "Registros del dia faltantes restantes:",
    data["registros_del_dia"].isna().sum()
)

data.loc[filas_registros_del_dia_null]

Registros del dia faltantes restantes: 77
Registros del dia faltantes restantes: 0


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.6000,93.2,5.3922,3.80030,...,2.6275,0.2874,6.2418,144,2009,5,NaN,January,N,-10.88
32,2009-02-02,979.9558,976.36,982.79,2.2236,80.692600,67.7100,91.4,6.7588,1.48330,...,0.5206,0.7330,54.6159,144,2009,33,invierno,February,NE,NaN
50,2009-02-20,996.7133,995.74,998.82,0.9071,94.240300,85.4000,97.9,2.9842,1.84030,...,-0.8950,-1.1091,231.0953,144,2009,51,invierno,February,SW,2.06
68,2009-03-10,981.9172,977.33,985.07,3.0183,80.567800,69.4600,93.5,7.0047,3.60120,...,-2.8383,-2.0031,215.2126,144,2009,69,primavera,March,SW,6.06
78,2009-03-20,1004.0028,1001.98,1005.52,1.0544,62.143400,46.6200,72.6,6.5535,2.08010,...,1.4188,0.8023,29.4868,144,2009,79,primavera,March,NE,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2416,2015-08-14,982.2497,979.68,985.56,1.5929,65.754400,34.5700,94.2,18.7542,1.85630,...,0.4244,0.1318,17.2560,144,2015,226,verano,July,N,26.51
2431,2015-08-29,996.4576,994.64,998.44,1.2345,80.705000,62.0100,99.4,14.7227,4.87512,...,-0.0739,0.0752,134.4735,144,2015,241,invierno,November,SE,33.72
2436,2015-09-03,987.6869,986.70,988.99,0.5961,86.616000,69.4100,96.3,5.8998,4.50396,...,-0.6178,-0.2332,200.6818,144,2015,246,otono,May,S,17.78
2440,2015-09-07,995.5186,993.71,998.42,1.3908,78.132900,49.2400,94.2,14.2828,NaN,...,-0.0173,-0.9815,268.9887,144,2015,250,otono,September,W,17.65


**presion_media**

Si *presion_media* es nula, pero *presion_min* y *presion_max* no lo son, entonces podemos estimar **presion_media** usando el promedio de *presion_min* y *presion_max*.

In [2314]:
filas_presion_media_nula = data["presion_media"].isna()

filas_presion_media_min_max = (
    filas_presion_media_nula
    & data["presion_max"].notna()
    & data["presion_min"].notna()
)

data.loc[filas_presion_media_min_max, "presion_media"] = (
    data.loc[filas_presion_media_min_max, "presion_min"]
    + data.loc[filas_presion_media_min_max, "presion_max"]
) / 2

print(
    "Presion media faltantes restantes:",
    data["presion_media"].isna().sum()
)

data.loc[filas_presion_media_nula, ["presion_media", "presion_min", "presion_max"]]

Presion media faltantes restantes: 2


,presion_media,presion_min,presion_max
21,977.510,971.48,983.54
72,991.345,988.47,994.22
165,988.005,985.18,990.83
167,995.625,991.66,999.59
174,991.350,988.50,994.20
...,...,...,...
2343,988.895,987.79,990.00
2363,982.145,978.09,986.20
2411,992.680,990.56,994.80
2418,983.330,982.39,984.27


Ahora, para los registros que no tienen dos de las columnas, simplemente tomaremos el promedio de *presion_media* de los 5 dias anteriores y los 5 dias posteriores.

In [2315]:
data_ordenada = data.sort_values("fecha").copy()
filas_faltantes = data_ordenada["presion_media"].isna()

for indice in data_ordenada.index[filas_faltantes]:
    posicion = data_ordenada.index.get_indexer([indice])[0]

    valores_vecinos = pd.concat([
        data_ordenada.iloc[max(0, posicion - 5):posicion]["presion_media"],
        data_ordenada.iloc[posicion + 1:posicion + 6]["presion_media"]
    ]).dropna()

    if not valores_vecinos.empty:
        data.loc[indice, "presion_media"] = valores_vecinos.mean()

print(
    "Presion media faltantes restantes:",
    data["presion_media"].isna().sum()
)


Presion media faltantes restantes: 0


**presion_min** y **presion_max**

En este caso, usamos la desviación estándar para encontrar presión_min y presión_max. Como suponemos normalidad en los valores de los datos tenemos que, según (1):
$$\text{presion min} \approx \max\{\text{presion media}-\text{presion desv} \sqrt{2 \ln(\text{registros del dia})},0\} \approx \max\{\text{presion max} - 2\text{presion desv} \sqrt{2 \ln(\text{registros del dia})},0\}$$

$$\text{presion max} \approx \max\{\text{presion media}+\text{presion desv} \sqrt{2 \ln(\text{registros del dia})},0\} \approx \max\{\text{presion min} + 2\text{presion desv} \sqrt{2 \ln(\text{registros del dia})},0\}$$

Iremos imputando de acuerdo a la información que tenemos hasta no tener datos nulos.

In [2316]:
filas_presion_min_nula = data["presion_min"].isna()

print(
    "Presion min faltantes restantes antes de imputar:",
    data["presion_min"].isna().sum()
)

filas_min_null_1 = (
    filas_presion_min_nula
    & data["presion_media"].notna()
    & data["presion_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_min_null_1, "presion_min"]= data.loc[filas_min_null_1, "presion_media"] - data.loc[filas_min_null_1, "presion_desv"]*np.sqrt(2* np.log(data.loc[filas_min_null_1, "registros_del_dia"]))
filas_presion_min_nula = data["presion_min"].isna()
print(
    "Presion min faltantes restantes en la primera imputación:",
    data["presion_min"].isna().sum()
)

filas_min_null_2 = (
    filas_presion_min_nula
    & data["presion_max"].notna()
    & data["presion_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de presion_dev.

data.loc[filas_min_null_2, "presion_min"]= data.loc[filas_min_null_2, "presion_media"] - (data["presion_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_min_null_2, "registros_del_dia"]))
filas_presion_min_nula = data["presion_min"].isna()

data.loc[:,"presion_min"]=data.loc[:,"presion_min"].clip(lower=0)
print(
    "Presion min faltantes restantes en la segunda imputación:",
    data["presion_min"].isna().sum()
)



Presion min faltantes restantes antes de imputar: 70
Presion min faltantes restantes en la primera imputación: 4
Presion min faltantes restantes en la segunda imputación: 0


Vemos que hemos llenado todos los valores de *presion_min* que estaban nulos. Ahora, haremos lo mismo para *presion_max*

In [2317]:
filas_presion_max_nula = data["presion_max"].isna()

print(
    "Presion max faltantes restantes antes de imputar:",
    data["presion_max"].isna().sum()
)

filas_max_null_1 = (
    filas_presion_max_nula
    & data["presion_media"].notna()
    & data["presion_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_max_null_1, "presion_max"]= data.loc[filas_max_null_1, "presion_media"] + data.loc[filas_max_null_1, "presion_desv"]*np.sqrt(2* np.log(data.loc[filas_max_null_1, "registros_del_dia"]))
filas_presion_max_nula = data["presion_max"].isna()
print(
    "Presion max faltantes restantes en la primera imputación:",
    data["presion_max"].isna().sum()
)

filas_max_null_2 = (
    filas_presion_max_nula
    & data["presion_min"].notna()
    & data["presion_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de presion_dev.

data.loc[filas_max_null_2, "presion_max"]= data.loc[filas_max_null_2, "presion_media"] + (data["presion_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_max_null_2, "registros_del_dia"]))
filas_presion_max_nula = data["presion_max"].isna()
data.loc[:,"presion_max"]=data.loc[:,"presion_max"].clip(lower=0)
print(
    "Presion min faltantes restantes en la segunda imputación:",
    data["presion_max"].isna().sum()
)


Presion max faltantes restantes antes de imputar: 63
Presion max faltantes restantes en la primera imputación: 0
Presion min faltantes restantes en la segunda imputación: 0


**presion_desv**

Ahora veremos que hacer con los valores de presion_desv nulos. Cómo ya los valores de presion_min, presion_max y presion_media están calculados, podemos tomar $$\max\left\{\text{presion desv} \approx \frac{\text{presion max} - \text{presion media}}{ \sqrt{2 \ln(\text{registros del dia})}},0\right\}$$

In [2318]:
filas_presion_desv_nula = data["presion_desv"].isna()

print(
    "Presion desv faltantes restantes antes de imputar:",
    data["presion_desv"].isna().sum()
)

data.loc[filas_presion_desv_nula,"presion_desv"] = (data.loc[filas_presion_desv_nula,"presion_max"]- data.loc[filas_presion_desv_nula,"presion_media"])/(np.sqrt(2*np.log(data.loc[filas_presion_desv_nula,"registros_del_dia"])))

data.loc[:,"presion_desv"]=data.loc[:,"presion_desv"].clip(lower=0)
print(
    "Presion desv faltantes restantes despues de imputar:",
    data["presion_desv"].isna().sum()
)

Presion desv faltantes restantes antes de imputar: 78
Presion desv faltantes restantes despues de imputar: 0


**humedad_media**, **humedad_max**, **humedad_min**, y **humedad_desv**

Seguimos el mismo procedimiento anterior para humedad.

In [2319]:
filas_humedad_media_nula = data["humedad_media"].isna()

filas_humedad_media_min_max = (
    filas_humedad_media_nula
    & data["humedad_max"].notna()
    & data["humedad_min"].notna()
)

data.loc[filas_humedad_media_min_max, "humedad_media"] = (
    data.loc[filas_humedad_media_min_max, "humedad_min"]
    + data.loc[filas_humedad_media_min_max, "humedad_max"]
) / 2

print(
    "humedad media faltantes restantes:",
    data["humedad_media"].isna().sum()
)

data.loc[data["humedad_media"].isna(), ["humedad_media", "humedad_min", "humedad_max"]]

humedad media faltantes restantes: 2


,humedad_media,humedad_min,humedad_max
444,NaN,67.23,NaN
1927,NaN,NaN,93.8


In [2320]:
data_ordenada = data.sort_values("fecha").copy()
filas_faltantes = data_ordenada["humedad_media"].isna()

for indice in data_ordenada.index[filas_faltantes]:
    posicion = data_ordenada.index.get_indexer([indice])[0]

    valores_vecinos = pd.concat([
        data_ordenada.iloc[max(0, posicion - 5):posicion]["humedad_media"],
        data_ordenada.iloc[posicion + 1:posicion + 6]["humedad_media"]
    ]).dropna()

    if not valores_vecinos.empty:
        data.loc[indice, "humedad_media"] = valores_vecinos.mean()
data.loc[:,"humedad_media"]=data.loc[:,"humedad_media"].clip(lower=0)
print(
    "humedad media faltantes restantes:",
    data["humedad_media"].isna().sum()
)

humedad media faltantes restantes: 0


In [2321]:
filas_humedad_min_nula = data["humedad_min"].isna()

print(
    "humedad min faltantes restantes antes de imputar:",
    data["humedad_min"].isna().sum()
)

filas_min_null_1 = (
    filas_humedad_min_nula
    & data["humedad_media"].notna()
    & data["humedad_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_min_null_1, "humedad_min"]= data.loc[filas_min_null_1, "humedad_media"] - data.loc[filas_min_null_1, "humedad_desv"]*np.sqrt(2* np.log(data.loc[filas_min_null_1, "registros_del_dia"]))
filas_humedad_min_nula = data["humedad_min"].isna()
print(
    "humedad min faltantes restantes en la primera imputación:",
    data["humedad_min"].isna().sum()
)

filas_min_null_2 = (
    filas_humedad_min_nula
    & data["humedad_max"].notna()
    & data["humedad_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de humedad_dev.

data.loc[filas_min_null_2, "humedad_min"]= data.loc[filas_min_null_2, "humedad_media"] - (data["humedad_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_min_null_2, "registros_del_dia"]))
filas_humedad_min_nula = data["humedad_min"].isna()

data.loc[:,"humedad_min"]=data.loc[:,"humedad_min"].clip(lower=0)
print(
    "humedad min faltantes restantes en la segunda imputación:",
    data["humedad_min"].isna().sum()
)



humedad min faltantes restantes antes de imputar: 79
humedad min faltantes restantes en la primera imputación: 2
humedad min faltantes restantes en la segunda imputación: 0


In [2322]:
filas_humedad_max_nula = data["humedad_max"].isna()

print(
    "humedad max faltantes restantes antes de imputar:",
    data["humedad_max"].isna().sum()
)

filas_max_null_1 = (
    filas_humedad_max_nula
    & data["humedad_media"].notna()
    & data["humedad_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_max_null_1, "humedad_max"]= data.loc[filas_max_null_1, "humedad_media"] + data.loc[filas_max_null_1, "humedad_desv"]*np.sqrt(2* np.log(data.loc[filas_max_null_1, "registros_del_dia"]))
filas_humedad_max_nula = data["humedad_max"].isna()
print(
    "humedad max faltantes restantes en la primera imputación:",
    data["humedad_max"].isna().sum()
)

filas_max_null_2 = (
    filas_humedad_max_nula
    & data["humedad_min"].notna()
    & data["humedad_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de humedad_dev.

data.loc[filas_max_null_2, "humedad_max"]= data.loc[filas_max_null_2, "humedad_media"] + (data["humedad_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_max_null_2, "registros_del_dia"]))
filas_humedad_max_nula = data["humedad_max"].isna()
data.loc[:,"humedad_max"]=data.loc[:,"humedad_max"].clip(lower=0)
print(
    "humedad max faltantes restantes en la segunda imputación:",
    data["humedad_max"].isna().sum()
)


humedad max faltantes restantes antes de imputar: 74
humedad max faltantes restantes en la primera imputación: 3
humedad max faltantes restantes en la segunda imputación: 0


In [2323]:
filas_humedad_desv_nula = data["humedad_desv"].isna()

print(
    "humedad desv faltantes restantes antes de imputar:",
    data["humedad_desv"].isna().sum()
)

data.loc[filas_humedad_desv_nula,"humedad_desv"] = (data.loc[filas_humedad_desv_nula,"humedad_max"]- data.loc[filas_humedad_desv_nula,"humedad_media"])/(np.sqrt(2*np.log(data.loc[filas_humedad_desv_nula,"registros_del_dia"])))
data.loc[:,"humedad_desv"]=data.loc[:,"humedad_desv"].clip(lower=0)
print(
    "humedad desv faltantes restantes despues de imputar:",
    data["humedad_desv"].isna().sum()
)

humedad desv faltantes restantes antes de imputar: 59
humedad desv faltantes restantes despues de imputar: 0


**viento_media**, **viento_max**, **viento_min**, y **viento_desv**

Seguimos el mismo procedimiento anterior para viento.

In [2324]:
filas_viento_media_nula = data["viento_media"].isna()

filas_viento_media_min_max = (
    filas_viento_media_nula
    & data["viento_max"].notna()
    & data["viento_min"].notna()
)

data.loc[filas_viento_media_min_max, "viento_media"] = (
    data.loc[filas_viento_media_min_max, "viento_min"]
    + data.loc[filas_viento_media_min_max, "viento_max"]
) / 2

print(
    "viento media faltantes restantes:",
    data["viento_media"].isna().sum()
)

data.loc[data["viento_media"].isna(), ["viento_media", "viento_min", "viento_max"]]

viento media faltantes restantes: 4


,viento_media,viento_min,viento_max
551,NaN,0.240,NaN
1214,NaN,1.116,NaN
1573,NaN,0.380,NaN
1757,NaN,0.160,NaN


In [2325]:
data_ordenada = data.sort_values("fecha").copy()
filas_faltantes = data_ordenada["viento_media"].isna()

for indice in data_ordenada.index[filas_faltantes]:
    posicion = data_ordenada.index.get_indexer([indice])[0]

    valores_vecinos = pd.concat([
        data_ordenada.iloc[max(0, posicion - 5):posicion]["viento_media"],
        data_ordenada.iloc[posicion + 1:posicion + 6]["viento_media"]
    ]).dropna()

    if not valores_vecinos.empty:
        data.loc[indice, "viento_media"] = valores_vecinos.mean()
data.loc[:,"viento_media"]=data.loc[:,"viento_media"].clip(lower=0)
print(
    "viento media faltantes restantes:",
    data["viento_media"].isna().sum()
)

viento media faltantes restantes: 0


In [2326]:
filas_viento_min_nula = data["viento_min"].isna()

print(
    "viento min faltantes restantes antes de imputar:",
    data["viento_min"].isna().sum()
)

filas_min_null_1 = (
    filas_viento_min_nula
    & data["viento_media"].notna()
    & data["viento_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_min_null_1, "viento_min"]= data.loc[filas_min_null_1, "viento_media"] - data.loc[filas_min_null_1, "viento_desv"]*np.sqrt(2* np.log(data.loc[filas_min_null_1, "registros_del_dia"]))
filas_viento_min_nula = data["viento_min"].isna()
print(
    "viento min faltantes restantes en la primera imputación:",
    data["viento_min"].isna().sum()
)

filas_min_null_2 = (
    filas_viento_min_nula
    & data["viento_max"].notna()
    & data["viento_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de viento_dev.

data.loc[filas_min_null_2, "viento_min"]= data.loc[filas_min_null_2, "viento_media"] - (data["viento_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_min_null_2, "registros_del_dia"]))
filas_viento_min_nula = data["viento_min"].isna()

data.loc[:,"viento_min"]=data.loc[:,"viento_min"].clip(lower=0)
print(
    "viento min faltantes restantes en la segunda imputación:",
    data["viento_min"].isna().sum()
)

viento min faltantes restantes antes de imputar: 86
viento min faltantes restantes en la primera imputación: 3
viento min faltantes restantes en la segunda imputación: 0


In [2327]:
filas_viento_max_nula = data["viento_max"].isna()

print(
    "viento max faltantes restantes antes de imputar:",
    data["viento_max"].isna().sum()
)

filas_max_null_1 = (
    filas_viento_max_nula
    & data["viento_media"].notna()
    & data["viento_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_max_null_1, "viento_max"]= data.loc[filas_max_null_1, "viento_media"] + data.loc[filas_max_null_1, "viento_desv"]*np.sqrt(2* np.log(data.loc[filas_max_null_1, "registros_del_dia"]))
filas_viento_max_nula = data["viento_max"].isna()
print(
    "viento max faltantes restantes en la primera imputación:",
    data["viento_max"].isna().sum()
)

filas_max_null_2 = (
    filas_viento_max_nula
    & data["viento_min"].notna()
    & data["viento_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de viento_dev.

data.loc[filas_max_null_2, "viento_max"]= data.loc[filas_max_null_2, "viento_media"] + (data["viento_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_max_null_2, "registros_del_dia"]))
filas_viento_max_nula = data["viento_max"].isna()
data.loc[:,"viento_max"]=data.loc[:,"viento_max"].clip(lower=0)
print(
    "viento max faltantes restantes en la segunda imputación:",
    data["viento_max"].isna().sum()
)

viento max faltantes restantes antes de imputar: 80
viento max faltantes restantes en la primera imputación: 1
viento max faltantes restantes en la segunda imputación: 0


In [2328]:
filas_viento_desv_nula = data["viento_desv"].isna()

print(
    "viento desv faltantes restantes antes de imputar:",
    data["viento_desv"].isna().sum()
)

data.loc[filas_viento_desv_nula,"viento_desv"] = (data.loc[filas_viento_desv_nula,"viento_max"]- data.loc[filas_viento_desv_nula,"viento_media"])/(np.sqrt(2*np.log(data.loc[filas_viento_desv_nula,"registros_del_dia"])))

data.loc[:,"viento_desv"]=data.loc[:,"viento_desv"].clip(lower=0)
print(
    "viento desv faltantes restantes despues de imputar:",
    data["viento_desv"].isna().sum()
)

viento desv faltantes restantes antes de imputar: 69
viento desv faltantes restantes despues de imputar: 0


**rafaga_media**, **rafaga_max**, **rafaga_min**, y **rafaga_desv**

Seguimos el mismo procedimiento anterior para rafaga.

In [2329]:
data_ordenada = data.sort_values("fecha").copy()
filas_faltantes = data_ordenada["rafaga_media"].isna()

for indice in data_ordenada.index[filas_faltantes]:
    posicion = data_ordenada.index.get_indexer([indice])[0]

    valores_vecinos = pd.concat([
        data_ordenada.iloc[max(0, posicion - 5):posicion]["rafaga_media"],
        data_ordenada.iloc[posicion + 1:posicion + 6]["rafaga_media"]
    ]).dropna()

    if not valores_vecinos.empty:
        data.loc[indice, "rafaga_media"] = valores_vecinos.mean()
    
data.loc[:,"rafaga_media"]=data.loc[:,"rafaga_media"].clip(lower=0)

print(
    "rafaga media faltantes restantes:",
    data["rafaga_media"].isna().sum()
)

rafaga media faltantes restantes: 0


In [2330]:
filas_rafaga_min_nula = data["rafaga_min"].isna()

print(
    "rafaga min faltantes restantes antes de imputar:",
    data["rafaga_min"].isna().sum()
)

filas_min_null_1 = (
    filas_rafaga_min_nula
    & data["rafaga_media"].notna()
    & data["rafaga_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_min_null_1, "rafaga_min"]= data.loc[filas_min_null_1, "rafaga_media"] - data.loc[filas_min_null_1, "rafaga_desv"]*np.sqrt(2* np.log(data.loc[filas_min_null_1, "registros_del_dia"]))
filas_rafaga_min_nula = data["rafaga_min"].isna()
print(
    "rafaga min faltantes restantes en la primera imputación:",
    data["rafaga_min"].isna().sum()
)

filas_min_null_2 = (
    filas_rafaga_min_nula
    & data["rafaga_max"].notna()
    & data["rafaga_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de rafaga_dev.

data.loc[filas_min_null_2, "rafaga_min"]= data.loc[filas_min_null_2, "rafaga_media"] - (data["rafaga_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_min_null_2, "registros_del_dia"]))
filas_rafaga_min_nula = data["rafaga_min"].isna()

data.loc[:,"rafaga_min"]=data.loc[:,"rafaga_min"].clip(lower=0)
print(
    "rafaga min faltantes restantes en la segunda imputación:",
    data["rafaga_min"].isna().sum()
)

rafaga min faltantes restantes antes de imputar: 69
rafaga min faltantes restantes en la primera imputación: 5
rafaga min faltantes restantes en la segunda imputación: 0


In [2331]:
filas_rafaga_max_nula = data["rafaga_max"].isna()

print(
    "rafaga max faltantes restantes antes de imputar:",
    data["rafaga_max"].isna().sum()
)

filas_max_null_1 = (
    filas_rafaga_max_nula
    & data["rafaga_media"].notna()
    & data["rafaga_desv"].notna()
    & data["registros_del_dia"].notna()
) ##primero modificamos el valor cuando tenemos todos los demás valores de presión y registros del día

data.loc[filas_max_null_1, "rafaga_max"]= data.loc[filas_max_null_1, "rafaga_media"] + data.loc[filas_max_null_1, "rafaga_desv"]*np.sqrt(2* np.log(data.loc[filas_max_null_1, "registros_del_dia"]))
filas_rafaga_max_nula = data["rafaga_max"].isna()
print(
    "rafaga max faltantes restantes en la primera imputación:",
    data["rafaga_max"].isna().sum()
)

filas_max_null_2 = (
    filas_rafaga_max_nula
    & data["rafaga_min"].notna()
    & data["rafaga_media"].notna()
    & data["registros_del_dia"].notna()
)#luego modificamos el valor si tampoco se conocen la desviación estandar. En ese caso, tomamos la mediana de rafaga_dev.

data.loc[filas_max_null_2, "rafaga_max"]= data.loc[filas_max_null_2, "rafaga_media"] + (data["rafaga_desv"]).median()*np.sqrt(2* np.log(data.loc[filas_max_null_2, "registros_del_dia"]))
filas_rafaga_max_nula = data["rafaga_max"].isna()

data.loc[:,"rafaga_max"]=data.loc[:,"rafaga_max"].clip(lower=0)
print(
    "rafaga max faltantes restantes en la segunda imputación:",
    data["rafaga_max"].isna().sum()
)

rafaga max faltantes restantes antes de imputar: 70
rafaga max faltantes restantes en la primera imputación: 1
rafaga max faltantes restantes en la segunda imputación: 0


In [2332]:
filas_rafaga_desv_nula = data["rafaga_desv"].isna()

print(
    "rafaga desv faltantes restantes antes de imputar:",
    data["rafaga_desv"].isna().sum()
)

data.loc[filas_rafaga_desv_nula,"rafaga_desv"] = (data.loc[filas_rafaga_desv_nula,"rafaga_max"]- data.loc[filas_rafaga_desv_nula,"rafaga_media"])/(np.sqrt(2*np.log(data.loc[filas_rafaga_desv_nula,"registros_del_dia"])))
data.loc[:,"rafaga_desv"]=data.loc[:,"rafaga_desv"].clip(lower=0)
print(
    "rafaga desv faltantes restantes despues de imputar:",
    data["rafaga_desv"].isna().sum()
)

rafaga desv faltantes restantes antes de imputar: 77
rafaga desv faltantes restantes despues de imputar: 0


**viento_norte** y **viento_este**

Para imputar los valores faltantes usamos la velocidad media y el ángulo de `direccion_viento`, medido desde el norte en sentido horario:

$$\text{viento\_norte} = \text{viento\_media} \cos(\text{direccion\_viento})$$

$$\text{viento\_este} = \text{viento\_media} \sin(\text{direccion\_viento})$$

En cuatro registros no se pueden calcular las componentes porque `direccion_viento` también es nula. Sin el ángulo no conocemos la orientación exacta del viento. Aunque `sector_viento` está disponible, solo indica un intervalo de 45 grados, por lo que usar su punto medio sería una aproximación y podría introducir error.

Por esta razón, lo más adecuado es conservar esos cuatro valores como faltantes y no inventar una dirección exacta. Si el modelo requiere una tabla completa, podemos usar el punto medio de `sector_viento` como segunda imputación, dejando explícito que esos valores tienen mayor incertidumbre. Otra opción es excluir únicamente esas filas del análisis que use las componentes del viento; no es necesario eliminar todo el conjunto de datos.

In [2333]:
columnas_viento = ["viento_norte", "viento_este"]
faltantes_antes = data[columnas_viento].isna().sum()

angulo = np.deg2rad(data["direccion_viento"].to_numpy())
velocidad = data["viento_media"].to_numpy()

componentes = {
    "viento_norte": velocidad * np.cos(angulo),
    "viento_este": velocidad * np.sin(angulo),
}

for columna in columnas_viento:
    faltantes = data[columna].isna()
    data.loc[faltantes, columna] = componentes[columna][faltantes.to_numpy()]

faltantes_despues = data[columnas_viento].isna().sum()

print("Faltantes antes de imputar:")
print(faltantes_antes)
print("\nFaltantes después de imputar:")
print(faltantes_despues)


Faltantes antes de imputar:
viento_norte    75
viento_este     59
dtype: int64

Faltantes después de imputar:
viento_norte    3
viento_este     1
dtype: int64


In [2334]:
# En estos registros falta el ángulo exacto, pero conocemos el sector del viento.
# Usamos el punto medio del sector como una aproximación de la dirección.
angulos_sector = {
    "N": 0,
    "NE": 45,
    "E": 90,
    "SE": 135,
    "S": 180,
    "SW": 225,
    "W": 270,
    "NW": 315,
}

angulo_sector = np.deg2rad(data["sector_viento"].map(angulos_sector).to_numpy())
velocidad = data["viento_media"].to_numpy()

for columna, factor in zip(
    ["viento_norte", "viento_este"],
    [np.cos(angulo_sector), np.sin(angulo_sector)]
):
    faltantes = data[columna].isna() & data["viento_media"].notna() & ~np.isnan(angulo_sector)
    data.loc[faltantes, columna] = velocidad[faltantes.to_numpy()] * factor[faltantes.to_numpy()]

print("Faltantes después de la segunda imputación:")
print(data[["viento_norte", "viento_este"]].isna().sum())

casos_sin_imputar = data.loc[
    data[["viento_norte", "viento_este"]].isna().any(axis=1),
    ["fecha", "viento_norte", "viento_este", "sector_viento", "viento_media"]
]
casos_sin_imputar

Faltantes después de la segunda imputación:
viento_norte    0
viento_este     0
dtype: int64


,fecha,viento_norte,viento_este,sector_viento,viento_media


**direccion_viento**

Para imputar los valores faltantes usamos las componentes del viento que ya fueron calculadas. La función *arctan2* permite recuperar el cuadrante correcto a partir de *viento_este* y *viento_norte*:

$$\text{direccion\_viento} = \operatorname{arctan2}(\text{viento\_este}, \text{viento\_norte})$$

El resultado se convierte de radianes a grados y se aplica el módulo 360 para garantizar que todas las direcciones estén en el intervalo válido `[0, 360)`. Este método es coherente con la representación vectorial del viento y conserva el sentido de sus componentes.

In [2335]:
faltantes_antes = data["direccion_viento"].isna().sum()

componentes_completas = (
    data["viento_norte"].notna()
    & data["viento_este"].notna()
)

angulo_calculado = np.degrees(
    np.arctan2(data["viento_este"], data["viento_norte"])
) % 360

faltantes = data["direccion_viento"].isna() & componentes_completas
data.loc[faltantes, "direccion_viento"] = angulo_calculado[faltantes]

print("Direcciones faltantes antes de imputar:", faltantes_antes)
print("Direcciones faltantes después de imputar:", data["direccion_viento"].isna().sum())
print("Direcciones fuera del rango [0, 360):", ((data["direccion_viento"] < 0) | (data["direccion_viento"] >= 360)).sum())

Direcciones faltantes antes de imputar: 77
Direcciones faltantes después de imputar: 0
Direcciones fuera del rango [0, 360): 0


**dia_del_anio**, **anio**, **mes** y **estacion_anio**

Imputamos estos valores con las funciones que definimos anteriormente para lidiar con valores no consistentes. También usamos la fecha para definir estacion_anio.

In [2336]:
columnas_fecha = ["anio", "dia_del_anio", "mes", "estacion_anio"]
faltantes_antes = data[columnas_fecha].isna().sum()

meses = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December",
}
estaciones = {
    12: "invierno", 1: "invierno", 2: "invierno",
    3: "primavera", 4: "primavera", 5: "primavera",
    6: "verano", 7: "verano", 8: "verano",
    9: "otono", 10: "otono", 11: "otono",
}

mask = data["fecha"].notna()
fechas = data.loc[mask, "fecha"]

data.loc[mask, "anio"] = data.loc[mask, "anio"].fillna(fechas.dt.year)
data.loc[mask, "dia_del_anio"] = data.loc[mask, "dia_del_anio"].fillna(fechas.dt.dayofyear)
data.loc[mask, "mes"] = data.loc[mask, "mes"].fillna(fechas.dt.month.map(meses))
data.loc[mask, "estacion_anio"] = data.loc[mask, "estacion_anio"].fillna(fechas.dt.month.map(estaciones))

data["anio"] = data["anio"].astype("Int64")
data["dia_del_anio"] = data["dia_del_anio"].astype("Int64")
data["mes"] = data["mes"].astype("category")
data["estacion_anio"] = data["estacion_anio"].astype("category")

faltantes_despues = data[columnas_fecha].isna().sum()

print("Faltantes antes de imputar:")
print(faltantes_antes)
print("\nFaltantes después de imputar:")
print(faltantes_despues)

Faltantes antes de imputar:
anio              83
dia_del_anio      66
mes               87
estacion_anio    321
dtype: int64

Faltantes después de imputar:
anio             0
dia_del_anio     0
mes              0
estacion_anio    0
dtype: int64


**sector_viento**

Ahora usamos la direccion para imputar los datos de sector viento faltantes.

In [2337]:
faltantes_antes = data["sector_viento"].isna().sum()

mask = data["sector_viento"].isna() & data["direccion_viento"].notna()
sectores_imputados = data.loc[mask, "direccion_viento"].apply(obtener_sector_compass)
data.loc[mask, "sector_viento"] = sectores_imputados

data["sector_viento"] = data["sector_viento"].astype("category")
faltantes_despues = data["sector_viento"].isna().sum()

print("Sectores faltantes antes de imputar:", faltantes_antes)
print("Sectores faltantes después de imputar:", faltantes_despues)



Sectores faltantes antes de imputar: 69
Sectores faltantes después de imputar: 0


**temperatura_max_manana**

Para los valores faltantes de esta variable, tomamos la temperatura promedio de los 5 días anteriores y 5 días posteriores.


In [2338]:
faltantes_antes = data["temp_max_manana"].isna().sum()

data_ordenada = data.sort_values("fecha").copy()

promedio_vecinos = (
    data_ordenada["temp_max_manana"]
    .rolling(window=11, center=True, min_periods=1)
    .mean()
)

mascara_nulos = data_ordenada["temp_max_manana"].isna()
data.loc[data_ordenada.index[mascara_nulos], "temp_max_manana"] = promedio_vecinos[mascara_nulos]

faltantes_despues = data["temp_max_manana"].isna().sum()

print("Temperaturas faltantes antes de imputar:", faltantes_antes)
print("Temperaturas faltantes después de imputar:", faltantes_despues)

Temperaturas faltantes antes de imputar: 93
Temperaturas faltantes después de imputar: 0


**Conclusión**

Finalmente confirmaremos que el dataset no tiene registros nulos.

In [2339]:
olumns = data.columns.tolist() #obtenemos la lista de columnas del dataset

for column in columns:
    nulos = data[data[column].isna()].copy() #mostramos los valores nulos de cada columna del dataset
    print(f"Columna: {column}, Números de datos nulos: {len(nulos)}, Proporción de datos nulos: {len(nulos)/len(data)*100:.2f}%")

Columna: fecha, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: presion_media, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: presion_min, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: presion_max, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: presion_desv, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: humedad_media, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: humedad_min, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: humedad_max, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: humedad_desv, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: viento_media, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: viento_min, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: viento_max, Números de datos nulos: 0, Proporción de datos nulos: 0.00%
Columna: viento_d

In [2340]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2486 entries, 0 to 2558
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   fecha              2486 non-null   datetime64[ns]
 1   presion_media      2486 non-null   float64       
 2   presion_min        2486 non-null   float64       
 3   presion_max        2486 non-null   float64       
 4   presion_desv       2486 non-null   float64       
 5   humedad_media      2486 non-null   float64       
 6   humedad_min        2486 non-null   float64       
 7   humedad_max        2486 non-null   float64       
 8   humedad_desv       2486 non-null   float64       
 9   viento_media       2486 non-null   float64       
 10  viento_min         2486 non-null   float64       
 11  viento_max         2486 non-null   float64       
 12  viento_desv        2486 non-null   float64       
 13  rafaga_media       2486 non-null   float64       
 14  rafaga_min   

In [2341]:
data

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.500000,1000.87,1.3993,0.910860,0.875000,94.800000,1.7650,0.77860,...,-0.3618,-0.0223,183.5308,143,2009,1,invierno,January,S,-2.120
1,2009-01-02,999.6006,997.930000,1002.65,1.5039,91.450000,86.600000,96.300000,2.7588,1.41950,...,0.4267,0.3689,40.8436,144,2009,2,invierno,July,NE,-0.820
2,2009-01-03,998.5486,993.050000,1002.49,3.1304,76.458100,48.390000,93.900000,15.1796,1.25090,...,-0.6993,-0.5268,216.9916,144,2009,3,invierno,January,SW,-0.630
3,2009-01-04,988.5107,985.120000,992.93,2.3223,89.417400,75.260438,97.946704,4.4904,1.72040,...,-1.1268,-1.0413,222.7419,144,2009,4,invierno,January,SW,-1.440
4,2009-01-05,990.4057,977.064977,997.54,4.2315,86.260400,74.600000,93.200000,5.3922,3.80030,...,2.6275,0.2874,6.2418,144,2009,5,invierno,January,N,-10.880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2552,2015-12-28,1003.5725,1002.350000,1004.96,0.7294,87.109700,72.100000,97.700000,7.5170,3.02724,...,-0.6644,-0.0853,187.3193,144,2015,362,verano,March,S,9.160
2553,2015-12-29,1002.1544,1000.830000,1005.33,1.2520,95.091000,79.400000,99.500000,5.3604,0.86610,...,-0.5544,-0.0871,188.9267,144,2015,363,invierno,December,S,5.650
2554,2015-12-30,1002.9883,997.510000,1005.73,2.5524,84.473700,43.624887,99.400000,12.9567,2.12740,...,-2.0540,-0.1130,183.1497,144,2015,364,invierno,December,S,2.920
2555,2015-12-31,996.9224,995.010000,999.20,1.4003,0.726064,0.495600,97.400000,17.6803,3.11310,...,-3.0401,0.1673,176.8493,144,2015,365,invierno,December,S,2.720


## 3 Construcción del modelo de interpolación lineal

Ahora, con el conjunto de datos reconstruido podemos comenzar a construir nuestros modelos.

### 3.1 Ingeniería de características y preparación de datos

## Declaración de IA
### Usos de la IA registrados.

| Uso | Descripción | Prompt | Análisis crítico del resultado | Aportes propios del estudiante |
| -------- | -------- | -------- | -------- | -------- |
| Creación de la tabla de tipos estadísticos de datos. (Sección 1.3) | Se usó Claude Sonnet 5 para crear la tabla de tipos estadísticos de datos basados en el diccionario de datos. | "Basado en la siguiente tabla de Diccionario de datos, termina esta tabla en formato markdown con los tipos estadísticos de datos sus rangos y cualquier otra consideración pertinente. Mantén las explicaciones cortas y procura usar el mismo formato de las primeras dos columnas en toda la tabla." | El resultado es consistente con los tipos de datos en la vida real, y se muestran correctamente sus rangos. Se modificaron los rangos de temperatura máx de mañana para tener en cuenta días excepcionalmente fríos o calientes. | El estudiante creó y llenó las dos primeras filas de la tabla para darle a la IA un ejemplo de lo que debería hacer. Además, confirmó por separado que las columnas correspondían a su clasificación estadística, pues esto es importante para la compresión fundamental de los datos y sus representaciones.  | 



## Bibliografía

(1) Math Stack Exchange. (2013). Expected value of maximum and minimum of n normal random variables [Publicación en foro en línea]. Recuperado de https://math.stackexchange.com/questions/473229/expected-value-of-maximum-and-minimum-of-n-normal-random-variables